# NOV-T: does decision-token branching beat temperature in an LLM program search for TSP heuristics?

**Artifact `art_Xt1OfbyR7Mvy` (experiment, claim V4).** An LLM (**Qwen3-8B-FP8** on vLLM 0.11, thinking off) mutates
Python heuristics `select_next_node(current_node, destination_node, unvisited_nodes, distance_matrix)` for constructive
TSP (the ReEvo/EoH interface). The mutations happen inside a FunSearch-style 5-island search seeded with nearest
neighbour (NN). Every arm gets the same number of LLM calls (1,056 per arm and seed, seeds 0 and 1):

| arm | how the LLM samples a new program |
|---|---|
| **T1.0** | temperature 1.0, top-p 1 |
| **T1.6** | temperature 1.6, top-p 0.95 |
| **DTB** (decision-token branching) | force a rank-2..8 token at the **highest-entropy** code position of the first 48 body tokens, then continue at T 1 |
| **RPB** (placebo) | the same forcing at a **random** eligible position |
| **IDEA** | T 1, plus the instruction *"based on an idea different from the functions above"* |
| NOSEL | no selection: always prompted with NN only (comparator, seed 0) |

**Behaviour instrument.** Every valid program is run on 2,000 frozen *probe states* (partial tours). Its 2,000
next-city decisions form a name-blind behaviour vector. Programs are grouped by complete linkage at agreement ≥ 0.95,
and the pre-registered test **V2-T** compares *behaviour clusters per 1,000 calls* between arms.

**Results of the full run:** V2-T is **DISCONFIRMED**. DTB found no more behaviours than plain T1.0 (pooled DTB/T1.0
= 0.963; the kill condition was ≤ 1.0). **V1-T is SUPPORTED**: every arm's best program closes ≥ 20% of NN's held-out
gap at n = 200.

### What this notebook runs
The full run needs a GPU vLLM server (about 2 h of generation on an NVIDIA L4). This notebook therefore **replays a
logged slice of the run on CPU** with the original code: functions copied from `src/*.py` and organised by the stages
of `method.py`.

| `method.py` stage | section | what runs here |
|---|---|---|
| `stage0` | 1 | instance sets, LKH-3 references (from the log), the 2,000 probe states, NN through the sandbox |
| `search` | 2-5 | for logged LLM calls: re-render the exact prompt, **re-derive the DTB/RPB forced token** from the logged top-20 log-probs, replay the island RNG, rebuild the program and **re-execute it in the fork sandbox** (64 SELECTION instances + 2,000 probes) |
| `analyze` | 6-8 | behaviour clusters, an MDS map of behaviour space, the pre-registered verdict rules, held-out n = 200 check |
| `figures` | 9 | summary table and figures |

The data (`mini_demo_data.json`, 91 logged calls) holds the 16 **matched calls** 192–207 (round 6, seed 0) of each of
the 5 arms, the selection-best program of every arm-seed and NOSEL, and the full-run summary numbers used for comparison.
Only the LLM sampling itself is not re-run: its outputs come from the log.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT on Colab, always install (method.py and src/common.py log with loguru)
_pip('loguru==0.7.3')

# numpy, scipy, matplotlib, transformers (tokenizer only) — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0',
         'transformers==5.0.0', 'tokenizers==0.22.2', 'huggingface_hub==1.4.0')

In [ ]:
from __future__ import annotations

import os

# src/common.py: every evaluator / numeric process must be single-threaded (set before numpy is imported)
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import ast
import builtins
import hashlib
import json
import keyword
import math
import pickle
import random
import re
import resource
import select
import signal
import struct
import sys
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import scipy.special
from loguru import logger
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial import distance_matrix as _scipy_dm
from scipy.spatial.distance import pdist, squareform

import matplotlib.pyplot as plt

# method.py: loguru to stdout (the original also adds a rotating file sink logs/run.log; dropped in the notebook)
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

# Notebook only: Python 3.12 warns when a multi-threaded process (the Jupyter kernel) calls os.fork(). The sandbox
# forks one short-lived, single-purpose child per program, exactly as the original evaluator processes did.
warnings.filterwarnings("ignore", message=r".*use of fork\(\) may lead to deadlocks.*")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-295f8e-prompt-bans-substitute-not-just-subtract/fork/run_5eGZGmskluur/round-10/experiment-53/demo/mini_demo_data.json"
import http.client
import json
import urllib.request
from pathlib import Path

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=15) as response:
            return json.loads(response.read().decode())
    # Network errors, timeouts, a placeholder URL, bad JSON, a cut-off response.
    except (OSError, ValueError, http.client.HTTPException) as e:
        print("GitHub data unavailable, using local mini_demo_data.json:", e)
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
EXAMPLES = data["examples"]   # logged calls (dataset novt_calls)
CTX = data["context"]         # LKH-3 references, NN baseline, full-run summary numbers
print(data["description"])
print(f"{len(EXAMPLES)} logged calls:", dict(Counter((e["group"], e["arm"]) for e in EXAMPLES)))
print("logged labels:", dict(Counter(e["label"] for e in EXAMPLES)))

## Configuration
All demo-scale knobs live in the next cell. The **measurement itself is never scaled down** (64 SELECTION instances,
2,000 probe states, linkage threshold 0.05), so every replayed number can be compared one-to-one with the logged run.
Only *how much* of the log is replayed changes.

In [ ]:
# Demo scale (how much of the logged run is replayed). The pipeline itself is not scaled.
N_CALLS_PER_ARM = 16     # matched logged calls replayed per arm (seed 0, round 6, calls 192..); max 16 in the demo data
                         # (original run: 1,056 calls per arm-seed, seeds 0 and 1 -> 10,812 calls incl. NOSEL)
N_BEST_PROGRAMS = 11     # arm-seed best programs re-evaluated (seed-0 bests, then seed-1, then NOSEL); max 11
N_HELDOUT = 32           # held-out n=200 instances for the V1-T check (original: 32, plus 96 extended + 13 TSPLIB)
N_HELDOUT_PROGRAMS = 5   # how many of the re-evaluated best programs also run on the held-out instances (NN always)
N_FORCINGS = 36          # logged DTB/RPB forcings re-derived from request-1 log probs, half each; max 36 (orig. test: 50)

## 1. Stage 0: instances, references, probe states, NN baseline

### 1.1 Instance sets (`src/tsp_task.py`)
Every instance is drawn uniformly from the unit square with a fixed seed, so the sets are rebuilt here exactly as in
the run:
- **SELECTION** (32 instances at n = 50 and 32 at n = 100) is the only set the search ever sees.
- **HELDOUT** (32 at n = 200) decides V1-T.
- The 20 **PROBE** instances (n = 50…200) carry the behaviour instrument.

The original caches the coordinates in `data/instances/*.npz`; here they are built in memory.

In [ ]:
# ---- src/tsp_task.py (instance sets, distance matrices) ----
# Instance sets (seed, n). SELECTION is the only set the search ever sees.
SETS: dict[str, list[tuple[int, int]]] = {
    "selection": [(910001 + i, 50) for i in range(32)] + [(910101 + i, 100) for i in range(32)],
    "heldout": [(920001 + i, 200) for i in range(32)],
    "heldout_ext": [(920033 + i, 200) for i in range(96)],
    "probe": [(930001 + i, int(round(50 + 150 * i / 19))) for i in range(20)],
}


def coords_for(seed: int, n: int) -> np.ndarray:
    return np.random.default_rng(seed).uniform(0, 1, (n, 2))


def dist_matrix(coords: np.ndarray) -> np.ndarray:
    return _scipy_dm(coords, coords).astype(np.float64)


def tour_length(tour, D: np.ndarray) -> float:
    t = np.asarray(tour, dtype=np.int64)
    return float(D[t, np.roll(t, -1)].sum())


def build_sets() -> dict[str, dict]:
    """Create every random instance set; returns {set: {'seeds','ns','coords'}}.

    Notebook: built in memory (the original also writes/reloads data/instances/<set>.npz and records its sha256)."""
    out = {}
    for name, spec in SETS.items():
        coords = [coords_for(s, n) for s, n in spec]
        for (s, n), c in zip(spec, coords):
            assert c.shape == (n, 2) and np.allclose(c, coords_for(s, n)), f"instance {s} mismatch"
        out[name] = {"seeds": [s for s, _ in spec], "ns": [n for _, n in spec], "coords": coords}
    return out


def set_matrices(sets: dict, name: str) -> list[np.ndarray]:
    return [dist_matrix(c) for c in sets[name]["coords"]]


sets = build_sets()
for name in ("selection", "heldout", "heldout_ext", "probe"):
    print(f"{name:12s} {len(sets[name]['seeds']):3d} instances, n in {sorted(set(sets[name]['ns']))}")

### 1.2 Reference tours and the fitness F (`src/evaluator.py`)
The **gap** of a tour is `100 (L − L_ref) / L_ref`, where `L_ref` is the LKH-3 tour length (elkai, 10 runs). Stage 0
computes these references once (`data/references.jsonl`), and the demo data carries them so that no solver has to be
installed.

Fitness is `F = 0.5·G50/G50(NN) + 0.5·G100/G100(NN)`: the mean gap on the n = 50 and n = 100 halves of SELECTION,
normalised by nearest neighbour. **Lower is better and NN = 1.**

In [ ]:
# ---- src/evaluator.py (references and gaps) ----
def gaps_from_lengths(lengths, L_ref: np.ndarray) -> np.ndarray:
    L = np.asarray(lengths, dtype=np.float64)
    return 100.0 * (L - L_ref) / L_ref


def fitness(gaps64: np.ndarray, nn_g50: float, nn_g100: float) -> tuple[float, float, float]:
    """F = 0.5 G50/G50(NN) + 0.5 G100/G100(NN); lower is better, NN = 1."""
    g50 = float(np.mean(gaps64[:32]))
    g100 = float(np.mean(gaps64[32:64]))
    return 0.5 * g50 / nn_g50 + 0.5 * g100 / nn_g100, g50, g100


# load_references(): {set: array of L_ref} -- the LKH-3 tour lengths logged by stage0, here from the demo data
refs = {k: np.array(v) for k, v in CTX["references"].items()}
for k, v in refs.items():
    print(f"{k:10s} {len(v)} LKH-3 reference lengths, mean {v.mean():.4f}")

### 1.3 The behaviour instrument: 2,000 frozen probe states (`src/probes.py`)
For each of the 20 PROBE instances the code builds an NN tour and a random-insertion tour, then cuts 50 random prefixes
from each. A probe state is `(instance, current city, unvisited set)` with destination 0. A program's **behaviour
vector** is its next-city choice in all 2,000 states. Two programs that agree on ≥ 95% of the states count as the same
behaviour, however different their code looks: the instrument is *name-blind*.

In [ ]:
# ---- src/probes.py (frozen probe states) ----
N_PER_TOUR = 50


def nn_tour(D: np.ndarray) -> list[int]:
    n = len(D)
    tour, U = [0], np.ones(n, dtype=bool)
    U[0] = False
    cur = 0
    for _ in range(n - 1):
        d = np.where(U, D[cur], np.inf)
        nxt = int(np.argmin(d))
        tour.append(nxt)
        U[nxt] = False
        cur = nxt
    return tour


def random_insertion_tour(D: np.ndarray, rng: np.random.Generator) -> list[int]:
    """Random insertion: cities in random order, each inserted where it lengthens the tour least."""
    n = len(D)
    order = list(rng.permutation(n))
    tour = order[:2]
    for c in order[2:]:
        t = np.asarray(tour)
        nxt = np.roll(t, -1)
        inc = D[t, c] + D[c, nxt] - D[t, nxt]
        k = int(np.argmin(inc))
        tour.insert(k + 1, int(c))
    i0 = tour.index(0)
    return [int(x) for x in tour[i0:] + tour[:i0]]


def _prefix_lengths(rng: np.random.Generator, n: int) -> tuple[list[int], bool]:
    pool = np.arange(1, n - 1)  # 1..n-2
    if len(pool) >= N_PER_TOUR:
        return [int(x) for x in rng.choice(pool, size=N_PER_TOUR, replace=False)], False
    first = [int(x) for x in rng.permutation(pool)]
    extra = [int(x) for x in rng.choice(pool, size=N_PER_TOUR - len(first), replace=True)]
    return first + extra, True


def build_probe_states() -> dict:
    """Returns {'states': [(inst, cur, unvisited_list)], 'meta': [...], 'n': [...]}.

    Notebook: built in memory (the original caches it in data/probe_states.json and records its sha256)."""
    states, meta, padded = [], [], []
    for i, (seed, n) in enumerate(SETS["probe"]):
        D = dist_matrix(coords_for(seed, n))
        tours = {"nn": nn_tour(D), "ri": random_insertion_tour(D, np.random.default_rng(930100 + i))}
        rng = np.random.default_rng(930200 + i)
        for kind in ("nn", "ri"):
            tour = tours[kind]
            assert tour[0] == 0 and sorted(tour) == list(range(n))
            ts, pad = _prefix_lengths(rng, n)
            if pad:
                padded.append({"instance": i, "tour": kind, "n": n})
            for t in ts:
                prefix = tour[:t]
                unv = sorted(set(range(n)) - set(prefix))
                states.append([i, int(tour[t - 1]), unv])
                meta.append({"instance": i, "tour": kind, "t": t, "n": n})
    obj = {"states": states, "meta": meta, "n": [n for _, n in SETS["probe"]], "padded": padded}
    return obj


def probe_matrices() -> list[np.ndarray]:
    return [dist_matrix(coords_for(seed, n)) for seed, n in SETS["probe"]]


def agreement(a: np.ndarray, b: np.ndarray) -> float:
    """Share of probe states with the same decision (ERR = -1 only matches -1)."""
    return float(np.mean(np.asarray(a) == np.asarray(b)))


pst = build_probe_states()
print(f"{len(pst['states'])} probe states from {len(pst['n'])} instances (n = {pst['n'][0]}..{pst['n'][-1]}); "
      f"padded (n-2 < 50): {pst['padded']}")
i0, cur0, unv0 = pst["states"][0]
print(f"example state 0: instance {i0}, current city {cur0}, {len(unv0)} unvisited cities, destination 0")

### 1.4 The sandbox for LLM-written programs (`src/sandbox.py`)
Generated code is untrusted. Before anything runs, a **static AST check** keeps only whitelisted imports, function and
class definitions, and constant assignments, and it rejects forbidden names (`open`, `exec`, `os`, dunder attributes,
…). The **AST hash** (sha256 of the AST dump without docstrings) identifies a program independent of comments and
formatting. The run uses it as the evaluation-cache key and to index the stored decision vectors.

In [ ]:
# ---- src/sandbox.py (part 1: limits, static check, AST hash) ----
ALLOWED_IMPORTS = {"numpy", "math", "random", "heapq", "itertools", "collections", "functools", "typing", "bisect"}
FORBIDDEN_NAMES = {"open", "exec", "eval", "compile", "__import__", "globals", "locals", "vars", "input",
                   "breakpoint", "exit", "quit", "help", "memoryview", "__builtins__", "__loader__", "__spec__"}
FORBIDDEN_MODULE_NAMES = {"os", "sys", "subprocess", "socket", "importlib", "builtins", "shutil", "pathlib",
                          "ctypes", "multiprocessing", "threading", "signal", "resource", "gc", "inspect"}
SAFE_CALLS_IN_CONST = {"float", "int", "abs", "round", "tuple", "list", "dict", "set", "frozenset", "range",
                       "min", "max", "pow", "complex", "bool", "str", "len", "sum"}
ERROR_CLASSES = ["no_code", "syntax_error", "import_violation", "runtime_error", "invalid_return", "timeout",
                 "truncated"]
FUNC = "select_next_node"
SEL_CPU = 30          # s CPU for the 64 SELECTION instances together
PROBE_CPU = 30        # s CPU for the 2,000 probe calls
HARD_CPU = 62         # hard RLIMIT_CPU of the combined child
WALL_FACTOR = 1.5     # parent wall limit = 1.5 x CPU limit (per phase)
SET_CPU = 600         # s CPU per HELDOUT / EXT / TSPLIB set
MEM_LIMIT = 2 * 1024 ** 3
PROBE_ERR_MAX = 0.10  # PROBE-VALID requires at most 10% ERR decisions
ERR = -1


class StaticReject(Exception):
    def __init__(self, cls: str, reason: str):
        super().__init__(reason)
        self.cls, self.reason = cls, reason


def _is_const_expr(node: ast.AST) -> bool:
    for sub in ast.walk(node):
        if isinstance(sub, ast.Call):
            f = sub.func
            ok = (isinstance(f, ast.Name) and f.id in SAFE_CALLS_IN_CONST) or (
                isinstance(f, ast.Attribute) and isinstance(f.value, ast.Name) and f.value.id in ("np", "math", "numpy"))
            if not ok:
                return False
        elif isinstance(sub, (ast.Lambda, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp, ast.Await,
                              ast.Yield, ast.YieldFrom, ast.NamedExpr)):
            return False
    return True


def static_check(code: str) -> str:
    """Return sanitised source (top level: imports, defs, constant assignments) or raise StaticReject."""
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        raise StaticReject("syntax_error", f"{e.msg} line {e.lineno}") from None
    keep = []
    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            names = [a.name for a in node.names] if isinstance(node, ast.Import) else [node.module or ""]
            for nm in names:
                if nm.split(".")[0] not in ALLOWED_IMPORTS or (isinstance(node, ast.ImportFrom) and node.level):
                    raise StaticReject("import_violation", f"import {nm}")
            keep.append(node)
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            keep.append(node)
        elif isinstance(node, (ast.Assign, ast.AnnAssign)) and node.value is not None:
            tg = node.targets if isinstance(node, ast.Assign) else [node.target]
            if all(isinstance(t, (ast.Name, ast.Tuple)) for t in tg) and _is_const_expr(node.value):
                keep.append(node)
        # every other top-level statement (prints, tests, loops, if __name__ ...) is stripped
    for sub in ast.walk(ast.Module(body=keep, type_ignores=[])):
        if isinstance(sub, (ast.Import, ast.ImportFrom)):
            names = [a.name for a in sub.names] if isinstance(sub, ast.Import) else [sub.module or ""]
            for nm in names:
                if nm.split(".")[0] not in ALLOWED_IMPORTS or (isinstance(sub, ast.ImportFrom) and sub.level):
                    raise StaticReject("import_violation", f"import {nm}")
        elif isinstance(sub, ast.Name):
            if sub.id in FORBIDDEN_NAMES or sub.id in FORBIDDEN_MODULE_NAMES:
                raise StaticReject("import_violation", f"forbidden name {sub.id}")
        elif isinstance(sub, ast.Attribute):
            if sub.attr.startswith("__") and sub.attr not in ("__name__", "__init__", "__len__", "__lt__",
                                                              "__eq__", "__hash__", "__repr__", "__iter__"):
                raise StaticReject("import_violation", f"dunder attribute {sub.attr}")
        elif isinstance(sub, ast.Call) and isinstance(sub.func, ast.Name) and sub.func.id in ("getattr", "setattr",
                                                                                            "delattr", "hasattr"):
            for a in sub.args[1:2]:
                if not (isinstance(a, ast.Constant) and isinstance(a.value, str) and not a.value.startswith("_")):
                    raise StaticReject("import_violation", f"{sub.func.id} with non-literal/dunder name")
    if not any(isinstance(n, ast.FunctionDef) and n.name == FUNC for n in keep):
        raise StaticReject("no_code", f"no {FUNC} defined")
    return ast.unparse(ast.Module(body=keep, type_ignores=[]))


def strip_docstrings(tree: ast.AST) -> ast.AST:
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Module)):
            b = node.body
            if b and isinstance(b[0], ast.Expr) and isinstance(b[0].value, ast.Constant) and isinstance(
                    b[0].value.value, str):
                node.body = b[1:] or [ast.Pass()]
    return tree


def ast_hash(code: str) -> str | None:
    """sha256 of the AST dump with docstrings stripped (comments never reach the AST)."""
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return None
    return hashlib.sha256(ast.dump(strip_docstrings(tree), annotate_fields=False).encode()).hexdigest()


# what the static check does to a program that tries to escape
try:
    static_check("import os\ndef select_next_node(c, d, u, D):\n    return os.getpid()\n")
except StaticReject as e:
    print("rejected:", e.cls, "-", e.reason)

**Execution (child side).** The candidate is `exec`'d with restricted builtins and a guarded `__import__`, and it is
re-executed **fresh for every instance and every probe call**, so module-level state cannot leak between them. Each
call receives a fresh `set` and a fresh writable copy of the distance matrix (ReEvo `eval.py` semantics). `run_tours`
builds a closed tour from city 0 on each instance. `run_probes` records one decision per probe state; an exception or
an invalid return counts as `ERR = -1`.

Two notebook-only changes in `_child`, neither of which changes any result:
- the child's prints and warnings go to `/dev/null` (the run's evaluator processes wrote them to their logs);
- the 2 GB `RLIMIT_AS` cap is applied on top of the address space the child inherits from the Jupyter kernel.
  The original children forked from lean evaluator processes, but on Colab the kernel also has torch mapped,
  which alone could exceed an absolute 2 GB cap.

In [ ]:
# ---- src/sandbox.py (part 2: child-side execution) ----
_SBX = {"dir": None}


def _sbx_dir() -> str:
    """An empty working directory per evaluator process, created BEFORE the child's RLIMIT_FSIZE 0 (inside the
    workspace: scratch/sbx/<pid>); the child cannot write files there (FSIZE 0, no open/os)."""
    if _SBX["dir"] is None or not os.path.isdir(_SBX["dir"]):
        # notebook: relative to the working directory (the original resolves the workspace from __file__)
        root = os.path.join(os.getcwd(), "scratch", "sbx", str(os.getpid()))
        os.makedirs(root, exist_ok=True)
        _SBX["dir"] = root
    return _SBX["dir"]


class CpuLimit(BaseException):
    """Raised from the SIGXCPU handler; BaseException so candidate `except Exception` cannot swallow it."""


def _on_xcpu(signum, frame):  # noqa: ARG001
    raise CpuLimit()


def _guarded_import(name, globals=None, locals=None, fromlist=(), level=0):
    if level != 0 or name.split(".")[0] not in ALLOWED_IMPORTS:
        raise ImportError(f"import of {name} is not allowed")
    return builtins.__import__(name, globals, locals, fromlist, level)


def _safe_builtins() -> dict:
    sb = {k: getattr(builtins, k) for k in dir(builtins) if k not in FORBIDDEN_NAMES and not k.startswith("_")}
    sb["__import__"] = _guarded_import
    sb["__build_class__"] = builtins.__build_class__
    sb["__name__"] = "candidate"
    return sb


class InvalidReturn(Exception):
    pass


def _fresh_fn(code_obj):
    ns = {"__builtins__": _safe_builtins(), "__name__": "candidate"}
    exec(code_obj, ns)
    fn = ns.get(FUNC)
    if not callable(fn):
        raise InvalidReturn(f"{FUNC} not callable")
    return fn


def _as_city(x, U: set) -> int:
    """Accept ints, numpy ints and integral floats that are in the unvisited set."""
    if isinstance(x, (bool, np.bool_)) or x is None:
        raise InvalidReturn(f"returned {x!r}")
    if isinstance(x, (int, np.integer)):
        v = int(x)
    elif isinstance(x, (float, np.floating)):
        if not np.isfinite(x) or float(x) != int(x):
            raise InvalidReturn(f"non-integer {x!r}")
        v = int(x)
    elif isinstance(x, np.ndarray) and x.ndim == 0 and np.issubdtype(x.dtype, np.integer):
        v = int(x)
    else:
        raise InvalidReturn(f"returned type {type(x).__name__}")
    if v not in U:
        raise InvalidReturn(f"returned {v} not in unvisited")
    return v


def run_tours(code_obj, mats, seeds, tours_out: list | None = None) -> list[float]:
    """Build closed tours from city 0 on each instance (ReEvo protocol); returns lengths (raises on any failure).

    Every call gets a fresh set copy and a fresh writable matrix copy; np.random / random are seeded once per
    instance with the instance seed; the program is exec'd fresh per instance.
    """
    out = []
    for D, sd in zip(mats, seeds):
        fn = _fresh_fn(code_obj)
        n = D.shape[0]
        np.random.seed(sd % (2 ** 32))
        random.seed(sd)
        cur, tour, U = 0, [0], set(range(1, n))
        for _ in range(n - 1):
            nxt = _as_city(fn(cur, 0, set(U), D.copy()), U)
            tour.append(nxt)
            U.remove(nxt)
            cur = nxt
        t = np.asarray(tour)
        out.append(float(D[t, np.roll(t, -1)].sum()))
        if tours_out is not None:
            tours_out.append(tour)
    return out


def run_probes(code_obj, states, mats) -> list[int]:
    """One decision per probe state; ERR (-1) = exception or invalid return. Fresh exec per call; RNGs seeded
    with 930000+k before every call."""
    dec = []
    for k, (inst, cur, unv) in enumerate(states):
        try:
            fn = _fresh_fn(code_obj)
            np.random.seed((930000 + k) % (2 ** 32))
            random.seed(930000 + k)
            U = set(unv)
            dec.append(_as_city(fn(cur, 0, set(U), mats[inst].copy()), U))
        except Exception:  # noqa: BLE001 - any ordinary failure of the candidate is an ERR decision
            dec.append(ERR)
    return dec


def _send(wfd: int, obj) -> None:
    data = pickle.dumps(obj)
    os.write(wfd, struct.pack("<Q", len(data)))
    view, off = memoryview(data), 0
    while off < len(data):
        off += os.write(wfd, view[off:off + (1 << 20)])


def _classify(e: BaseException, phase: str) -> dict:
    if isinstance(e, CpuLimit):
        return {"ok": False, "cls": "probe_timeout" if phase == "probe" else "timeout", "reason": "cpu soft limit"}
    if isinstance(e, InvalidReturn):
        return {"ok": False, "cls": "invalid_return", "reason": str(e)[:300]}
    if isinstance(e, MemoryError):
        return {"ok": False, "cls": "runtime_error", "reason": f"MemoryError {e}"[:300]}
    if isinstance(e, ImportError):
        return {"ok": False, "cls": "import_violation", "reason": str(e)[:300]}
    if isinstance(e, RecursionError):
        return {"ok": False, "cls": "runtime_error", "reason": f"RecursionError {e}"[:300]}
    return {"ok": False, "cls": "runtime_error", "reason": f"{type(e).__name__}: {e}"[:300]}


def _child(wfd: int, code: str, jobs: list, cpu_soft: int, cpu_hard: int, extra_cpu: list) -> None:
    """jobs: list of (phase, mode, payload). Each phase's result is sent as its own frame; the first failure stops."""
    try:
        # notebook: keep candidate prints and numpy warnings out of the notebook output (the original evaluator
        # processes wrote them to their own log); results are unaffected
        warnings.simplefilter("ignore")
        sys.stdout = sys.stderr = open(os.devnull, "w")
        resource.setrlimit(resource.RLIMIT_CPU, (cpu_soft, cpu_hard))
        # notebook: the child is forked from the Jupyter kernel (numpy, matplotlib, the tokenizer and, on Colab, torch
        # already mapped) instead of a lean evaluator process, so the 2 GB cap is added to the inherited address space
        inherited = int(open("/proc/self/statm").read().split()[0]) * os.sysconf("SC_PAGE_SIZE")
        resource.setrlimit(resource.RLIMIT_AS, (inherited + MEM_LIMIT, inherited + MEM_LIMIT))
        resource.setrlimit(resource.RLIMIT_FSIZE, (0, 0))
        signal.signal(signal.SIGXCPU, _on_xcpu)
        try:
            os.nice(5)
        except OSError:
            pass
        os.chdir(_sbx_dir())
        code_obj = compile(code, "<candidate>", "exec")
    except BaseException as e:  # noqa: BLE001
        _send(wfd, {"phase": jobs[0][0], **_classify(e, jobs[0][0])})
        os._exit(0)
    for j, (phase, mode, payload) in enumerate(jobs):
        try:
            if j > 0:
                used = resource.getrusage(resource.RUSAGE_SELF)
                used_s = used.ru_utime + used.ru_stime
                soft = min(int(used_s) + 1 + extra_cpu[j], cpu_hard)
                resource.setrlimit(resource.RLIMIT_CPU, (soft, cpu_hard))
            t0 = time.process_time()
            if mode == "tours":
                mats, seeds = payload[0], payload[1]
                want = len(payload) > 2 and payload[2]
                tours: list = []
                res = {"ok": True, "lengths": run_tours(code_obj, mats, seeds, tours if want else None)}
                if want:
                    res["tours"] = tours
            elif mode == "probes":
                states, mats = payload
                res = {"ok": True, "decisions": run_probes(code_obj, states, mats)}
            else:
                raise ValueError(mode)
            res["cpu"] = time.process_time() - t0
            res["phase"] = phase
        except BaseException as e:  # noqa: BLE001 - candidate code may raise anything
            res = {"phase": phase, **_classify(e, phase)}
        try:
            _send(wfd, res)
        except BaseException:  # noqa: BLE001
            os._exit(0)
        if not res.get("ok"):
            break
    os._exit(0)

**Fork server (parent side).** `fork_eval` forks one child per program. The child sets `RLIMIT_CPU` (30 s for
SELECTION and 30 s for the probes), `RLIMIT_AS` (2 GB) and `RLIMIT_FSIZE` (0), then streams one length-prefixed
pickle frame per phase back through a pipe. The parent enforces a wall limit of 1.5 × the CPU limit.

In the run, spawned evaluator processes held the instance payload (`worker_init`) and called `evaluate_program`
(SELECTION + probes) or `evaluate_set` (held-out sets). Here the notebook process plays the role of one such evaluator.

In [ ]:
# ---- src/sandbox.py (part 3: fork server and evaluator-worker API) ----
def fork_eval(code: str, jobs: list, cpu_limits: list[int], cpu_hard: int | None = None) -> list[dict]:
    """Fork one child that runs `code` over `jobs` (phase, mode, payload) in order.

    cpu_limits[j] is the CPU budget of phase j (the soft limit is raised before every later phase);
    the parent's wall limit for phase j is WALL_FACTOR x cpu_limits[j]. Returns one dict per phase reached.
    """
    cpu_hard = cpu_hard or (sum(cpu_limits) + 2)
    _sbx_dir()
    rfd, wfd = os.pipe()
    pid = os.fork()
    if pid == 0:
        os.close(rfd)
        try:
            _child(wfd, code, jobs, cpu_limits[0], cpu_hard, cpu_limits)
        finally:
            os._exit(0)
    os.close(wfd)
    results: list[dict] = []
    buf = b""
    phase_idx = 0
    t_phase = time.time()
    deadline = t_phase + WALL_FACTOR * cpu_limits[0] + 2
    killed = False
    eof = False
    while not eof and phase_idx < len(jobs):
        left = deadline - time.time()
        if left <= 0:
            os.kill(pid, signal.SIGKILL)
            killed = True
            break
        r, _, _ = select.select([rfd], [], [], min(left, 1.0))
        if not r:
            continue
        b = os.read(rfd, 1 << 20)
        if not b:
            eof = True
        buf += b
        while len(buf) >= 8:
            (ln,) = struct.unpack("<Q", buf[:8])
            if len(buf) < 8 + ln:
                break
            res = pickle.loads(buf[8:8 + ln])
            buf = buf[8 + ln:]
            res["wall"] = time.time() - t_phase
            results.append(res)
            phase_idx += 1
            t_phase = time.time()
            if phase_idx < len(jobs):
                deadline = t_phase + WALL_FACTOR * cpu_limits[phase_idx] + 2
            if not res.get("ok"):
                phase_idx = len(jobs)
    os.close(rfd)
    _, status = os.waitpid(pid, 0)
    if len(results) < len(jobs) and (not results or results[-1].get("ok")):
        phase = jobs[len(results)][0]
        tcls = "probe_timeout" if phase == "probe" else "timeout"
        if killed:
            results.append({"ok": False, "cls": tcls, "reason": "wall limit", "phase": phase})
        elif os.WIFSIGNALED(status) and os.WTERMSIG(status) in (signal.SIGXCPU, signal.SIGKILL):
            results.append({"ok": False, "cls": tcls, "reason": f"cpu limit (signal {os.WTERMSIG(status)})",
                            "phase": phase})
        else:
            sig = os.WTERMSIG(status) if os.WIFSIGNALED(status) else None
            results.append({"ok": False, "cls": "runtime_error", "reason": f"child died (signal {sig})",
                            "phase": phase})
    return results


# ------------------------------------------------------------------ evaluator worker (spawned process)
_W: dict = {}


def worker_init(sets_payload: dict) -> None:
    """sets_payload: {'selection': (mats, seeds), 'heldout': ..., 'probe_states': [...], 'probe_mats': [...]}.

    Matrices stay writable: every candidate call receives its own D.copy(), so in-place writes cannot leak.
    """
    os.environ["OMP_NUM_THREADS"] = "1"
    _W.update(sets_payload)
    _sbx_dir()


def evaluate_program(raw_program: str, with_probes: bool = True) -> dict:
    """Static check + SELECTION (+ PROBES in the same child). Returns a dict with label, lengths, decisions."""
    try:
        code = static_check(raw_program)
    except StaticReject as e:
        return {"label": e.cls, "reason": e.reason, "lengths": None, "decisions": None, "probe_label": None,
                "cpu_sel": 0.0, "cpu_probe": 0.0, "wall": 0.0}
    jobs = [("selection", "tours", _W["selection"])]
    limits = [SEL_CPU]
    if with_probes:
        jobs.append(("probe", "probes", (_W["probe_states"], _W["probe_mats"])))
        limits.append(PROBE_CPU)
    t0 = time.time()
    res = fork_eval(code, jobs, limits, cpu_hard=HARD_CPU if with_probes else SEL_CPU + 2)
    out = {"label": None, "reason": None, "lengths": None, "decisions": None, "probe_label": None,
           "cpu_sel": None, "cpu_probe": None, "wall": time.time() - t0}
    r0 = res[0]
    if not r0.get("ok"):
        out.update(label=r0["cls"], reason=r0.get("reason"), cpu_sel=r0.get("cpu"))
        return out
    out.update(label="valid", lengths=r0["lengths"], cpu_sel=r0.get("cpu"))
    if with_probes:
        r1 = res[1] if len(res) > 1 else {"ok": False, "cls": "probe_timeout", "reason": "missing"}
        if r1.get("ok"):
            dec = r1["decisions"]
            n_err = sum(1 for d in dec if d == ERR)
            out.update(decisions=dec, cpu_probe=r1.get("cpu"), n_err=n_err,
                       probe_label="probe_valid" if n_err <= PROBE_ERR_MAX * len(dec) else "probe_err")
        else:
            out.update(probe_label=r1.get("cls", "probe_error"), reason=r1.get("reason"))
    return out


def evaluate_set(raw_program: str, set_name: str, cpu_limit: int = SET_CPU) -> dict:
    """Tours on a held-out set (HELDOUT / EXT / TSPLIB) in one child with `cpu_limit` seconds of CPU."""
    try:
        code = static_check(raw_program)
    except StaticReject as e:
        return {"label": e.cls, "reason": e.reason, "lengths": None}
    res = fork_eval(code, [(set_name, "tours", _W[set_name])], [cpu_limit])[0]
    if res.get("ok"):
        return {"label": "valid", "lengths": res["lengths"], "tours": res.get("tours"), "cpu": res.get("cpu"),
                "wall": res.get("wall")}
    return {"label": res["cls"], "reason": res.get("reason"), "lengths": None}


def probe_program(raw_program: str, cpu_limit: int = PROBE_CPU) -> dict:
    try:
        code = static_check(raw_program)
    except StaticReject as e:
        return {"label": e.cls, "decisions": None}
    res = fork_eval(code, [("probe", "probes", (_W["probe_states"], _W["probe_mats"]))], [cpu_limit])[0]
    if res.get("ok"):
        return {"label": "ok", "decisions": res["decisions"], "cpu": res.get("cpu")}
    return {"label": res["cls"], "reason": res.get("reason"), "decisions": None}

### 1.5 The seed program NN and the fitness normalisation (`src/baselines.py`, `src/stage0.py`)
Nearest neighbour is both the seed of every island and the baseline. Running it through the same sandbox gives
`G50(NN)` and `G100(NN)`, which normalise F (so F(NN) = 1). The cell compares them with the values stage 0 logged.

The evaluator payload holds SELECTION, the first `N_HELDOUT` held-out instances (the run held all 32, plus 96 extended
and 13 TSPLIB instances), and the probe states with their matrices.

In [ ]:
# ---- src/baselines.py: the seed program (nearest neighbour) ----
NN_PROGRAM = '''import numpy as np
import math


def select_next_node(current_node, destination_node, unvisited_nodes, distance_matrix):
    """Return the next city to visit: one element of unvisited_nodes."""
    next_node = None
    best_distance = float("inf")
    for node in unvisited_nodes:
        distance = distance_matrix[current_node][node]
        if distance < best_distance:
            best_distance = distance
            next_node = node
    return next_node
'''

# ---- src/evaluator.py load_payload(), restricted to what the demo needs ----
payload = {
    "selection": (set_matrices(sets, "selection"), list(sets["selection"]["seeds"])),
    "heldout": (set_matrices(sets, "heldout")[:N_HELDOUT], list(sets["heldout"]["seeds"])[:N_HELDOUT]),
    "probe_states": [(int(i), int(c), list(u)) for i, c, u in pst["states"]],
    "probe_mats": probe_matrices(),
}
worker_init(payload)

# ---- stage0: NN through the sandbox -> G50(NN), G100(NN) ----
t0 = time.time()
nn_res = evaluate_program(NN_PROGRAM, with_probes=True)
nn_sel_gaps = gaps_from_lengths(nn_res["lengths"], refs["selection"])
NN_G50, NN_G100 = float(np.mean(nn_sel_gaps[:32])), float(np.mean(nn_sel_gaps[32:64]))
nn_F = fitness(nn_sel_gaps, NN_G50, NN_G100)[0]
logged_nn = CTX["nn_baseline"]
print(f"NN: label={nn_res['label']}, {nn_res['probe_label']} (ERR decisions: {nn_res['n_err']}), "
      f"{time.time() - t0:.2f}s wall")
print(f"  G50(NN)  = {NN_G50:.4f}%   logged {logged_nn['G50']:.4f}%")
print(f"  G100(NN) = {NN_G100:.4f}%   logged {logged_nn['G100']:.4f}%")
print(f"  F(NN)    = {nn_F:.4f}")
print(f"  max |gap - logged gap| over the 64 SELECTION instances: "
      f"{np.max(np.abs(nn_sel_gaps - np.array(logged_nn['selection_gaps']))):.1e}")

## 2. Search, step 1: the prompt the LLM sees (`src/prompt.py`)
The search uses FunSearch-style **best-shot prompting**:
- The island database picks k = 2 parent programs (k = 1 while an island only holds NN), sorted with the **worse first**.
- The parents are rendered as `select_next_node_v0`, `select_next_node_v1` under a task docstring, and the model is
  asked to write the next version. The IDEA arm differs only in this last instruction.
- The assistant turn is **prefilled** with the new function header and docstring, so the model only writes the body.

The generated text is cut at the first stop string and trimmed FunSearch-style to the longest prefix that parses.

In [ ]:
# ---- src/prompt.py (prompt construction, prefill, program extraction with FunSearch's trim) ----
TASK_DOCSTRING = (
    "Builds a short closed tour for the Travelling Salesman Problem by greedy construction. The tour starts at "
    "city 0. At every step select_next_node is called with the current city, the destination city (city 0, where "
    "the tour ends), the set of unvisited cities and the n x n numpy array of Euclidean distances, and it returns "
    "the next city to visit, one element of unvisited_nodes. When no city is left the tour returns to city 0. "
    "Instances have 50 to 100 cities uniform in the unit square; shorter total tour length is better."
)
IMPORTS = "import numpy as np\nimport math\n"
SIG = "(current_node, destination_node, unvisited_nodes, distance_matrix):"
SPEC_DOC = "Return the next city to visit: one element of unvisited_nodes."
STOPS = ["```", "\ndef ", "\nclass ", "\nif __name__"]
MODEL_ID = "Qwen/Qwen3-8B-FP8"
MODEL_REV = "220b46e3b2180893580a4454f21f22d3ebb187d3"

_TOK = None


def tokenizer():
    global _TOK
    if _TOK is None:
        from transformers import AutoTokenizer

        _TOK = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REV)
    return _TOK


def _doc(i: int) -> str:
    return SPEC_DOC if i == 0 else f"Improved version of select_next_node_v{i - 1}."


def render_parent(body: str, i: int) -> str:
    return f'def select_next_node_v{i}{SIG}\n    """{_doc(i)}"""\n{body.rstrip()}\n'


def instruction(k: int, idea: bool) -> str:
    if not idea:
        return f"Write select_next_node_v{k}."
    if k == 1:
        return "Write select_next_node_v1 based on an idea different from the function above."
    return f"Write select_next_node_v{k} based on an idea different from both functions above."


def user_turn(parent_bodies: list[str], idea: bool = False) -> str:
    """parent_bodies sorted ascending by score (the WORSE first), already chosen by the database."""
    k = len(parent_bodies)
    parents = "\n\n".join(render_parent(b, i) for i, b in enumerate(parent_bodies))
    return f"{TASK_DOCSTRING}\n\n```python\n{IMPORTS}\n\n{parents}```\n\n{instruction(k, idea)}"


def prefill(k: int) -> str:
    return f'```python\ndef select_next_node_v{k}{SIG}\n    """Improved version of select_next_node_v{k - 1}."""\n'


def render_prompt(parent_bodies: list[str], idea: bool = False) -> tuple[str, list[int]]:
    """Full text (chat template, thinking off, plus the prefill) and its token ids (no special tokens added)."""
    tok = tokenizer()
    k = len(parent_bodies)
    text = tok.apply_chat_template([{"role": "user", "content": user_turn(parent_bodies, idea)}], tokenize=False,
                                   add_generation_prompt=True, enable_thinking=False)
    text = text + prefill(k)
    ids = tok(text, add_special_tokens=False)["input_ids"]
    return text, ids


def nosel_prompt() -> tuple[str, list[int]]:
    return render_prompt([nn_body(NN_PROGRAM)], idea=False)


def nn_body(program: str) -> str:
    """The body text (after the docstring, comments kept) of the program's select_next_node function."""
    tree = ast.parse(program)
    fn = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == "select_next_node")
    first = fn.body[0]
    if isinstance(first, ast.Expr) and isinstance(first.value, ast.Constant) and isinstance(first.value.value, str):
        start = first.end_lineno  # 1-based last line of the docstring
    else:
        start = fn.lineno  # header line (single-line header assumed)
    lines = program.split("\n")
    return "\n".join(lines[start:fn.end_lineno]).rstrip() + "\n"


def cut_at_stops(text: str) -> tuple[str, bool]:
    """Cut a generation at the first stop string (stops are also applied by the server)."""
    pos = [text.find(s) for s in STOPS if s in text]
    if pos:
        return text[:min(pos)], True
    return text, False


def program_from_body(body: str, k: int) -> str:
    return f'{IMPORTS}\n\ndef select_next_node{SIG}\n    """Improved version of select_next_node_v{k - 1}."""\n{body}'


def _n_body_statements(code: str) -> int:
    tree = ast.parse(code)
    fn = next((n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == "select_next_node"), None)
    if fn is None:
        return 0
    stmts = fn.body
    if stmts and isinstance(stmts[0], ast.Expr) and isinstance(stmts[0].value, ast.Constant):
        stmts = stmts[1:]
    return len(stmts)


def extract(generated: str, k: int) -> dict:
    """body = generated text up to the stop; program = imports + header + docstring + body; FunSearch trim."""
    body, _ = cut_at_stops(generated)
    body = body.rstrip() + "\n" if body.strip() else ""
    if not body.strip():
        return {"program": None, "body": "", "label": "no_code", "trimmed_lines": 0}
    lines = body.rstrip("\n").split("\n")
    for cut in range(len(lines), 0, -1):
        b = "\n".join(lines[:cut]) + "\n"
        prog = program_from_body(b, k)
        try:
            ok = _n_body_statements(prog) >= 1
        except SyntaxError:
            continue
        if ok:
            return {"program": prog, "body": b, "label": None, "trimmed_lines": len(lines) - cut}
    return {"program": None, "body": body, "label": "syntax_error", "trimmed_lines": len(lines)}

### 2.1 Re-rendering logged prompts
The run logged the sha256 of every prompt. The next cell loads only the model's **tokenizer** (a few MB from the
Hugging Face Hub, no weights) and re-renders each logged call's chat-templated prompt from its parent programs. It
then checks the hash and the token count against the log, and prints one IDEA prompt in full.

In [ ]:
tok = tokenizer()   # Qwen/Qwen3-8B-FP8 @ 220b46e, tokenizer files only


def sha256_text(s: str) -> str:   # src/common.py
    return hashlib.sha256(s.encode("utf-8")).hexdigest()


def parent_bodies_of(e: dict) -> list[str]:
    """Parent bodies of a logged call in prompt order (worse first); a None entry is the seed program NN."""
    return [b if b is not None else nn_body(NN_PROGRAM) for b in e["parent_bodies"]]


e_show = next(e for e in EXAMPLES if e["arm"] == "IDEA" and e["group"] == "block" and e["k"] == 2)
print(f"===== user turn of IDEA|{e_show['seed']}|{e_show['call_idx']} (parents {e_show['parent_ids']}, "
      f"parent F {[round(f, 3) for f in e_show['parent_F']]}) =====")
print(user_turn(parent_bodies_of(e_show), idea=True))
print("===== assistant prefill (the model continues from here) =====")
print(prefill(e_show["k"]))
print("non-IDEA arms end with:", repr(instruction(2, False)))

n_hash = n_len = n_with = 0
for e in EXAMPLES:
    text, ids = render_prompt(parent_bodies_of(e), idea=(e["arm"] == "IDEA"))
    if e["prompt_sha256"]:
        n_with += 1
        n_hash += sha256_text(text) == e["prompt_sha256"]
        n_len += len(ids) == e["prompt_len_tokens"]
print(f"\nre-rendered prompts: sha256 identical to the log {n_hash}/{n_with}, token count identical {n_len}/{n_with}")

## 3. Search, step 2: decision-token branching (`src/dtb.py`)
For DTB and RPB, **request 1** asks vLLM for the first 48 body tokens at T = 1 together with the top-20 log
probabilities at each position. Then:
1. **Eligible positions** are tokens that are not whitespace-only and whose first character lies in *code*, not inside
   a comment or string. A small Python-aware lexer tracks this.
2. **Position j\***: DTB takes the eligible position with the **highest entropy** of the renormalised top-20, i.e. the
   model's most uncertain code decision. RPB draws j\* **uniformly** (the placebo).
3. **Forced token**: drawn uniformly from ranks 2–8 at j\*. Special tokens, backticks, whitespace and tokens that would
   close the function are filtered out.
4. **Request 2** continues at T = 1 from `prompt + body[:j*] + forced token`.

Each draw uses its own seeded RNG stream (`[seed, arm, call, 3|4]`), so every forcing can be re-derived exactly from
the logged log probs.

In [ ]:
# ---- src/dtb.py (lexer, entropy, position and token rules) ----
CODE, COMMENT, STR_S, STR_D, STR_TS, STR_TD = range(6)
SPECIAL_MIN_ID = 151643  # every id >= this is a special / added token in the Qwen3 vocabulary
ARM_CODE = {"T1.0": 0, "T1.6": 1, "DTB": 2, "RPB": 3, "IDEA": 4, "NOSEL": 5}


def lex_states(text: str) -> list[int]:
    """State BEFORE each character of `text` (Python-aware: code / comment / 4 string kinds, with escapes)."""
    st, out, i, n = CODE, [], 0, len(text)
    while i < n:
        ch = text[i]
        if st == CODE:
            out.append(CODE)
            if ch == "#":
                st = COMMENT
            elif ch in "'\"":
                if text[i:i + 3] in ("'''", '"""'):
                    out.extend([CODE, CODE])
                    st = STR_TS if ch == "'" else STR_TD
                    i += 3
                    continue
                st = STR_S if ch == "'" else STR_D
            i += 1
        elif st == COMMENT:
            out.append(COMMENT)
            if ch == "\n":
                st = CODE
            i += 1
        else:
            out.append(st)
            if ch == "\\" and i + 1 < n:
                out.append(st)
                i += 2
                continue
            if st == STR_S and ch == "'" or st == STR_D and ch == '"':
                st = CODE
            elif st in (STR_S, STR_D) and ch == "\n":
                st = CODE  # unterminated single-line string: recover at the line end
            elif st == STR_TS and text[i:i + 3] == "'''" or st == STR_TD and text[i:i + 3] == '"""':
                out.extend([st, st])
                st = CODE
                i += 3
                continue
            i += 1
    return out[:n]


def token_offsets(tok, ids: list[int]) -> tuple[list[str], list[int], str]:
    """Incremental decode: per-token text, start offsets and the full decoded text of the id sequence."""
    texts, starts, prev = [], [], ""
    for k in range(len(ids)):
        cur = tok.decode(ids[:k + 1], skip_special_tokens=False, clean_up_tokenization_spaces=False)
        starts.append(len(prev))
        texts.append(cur[len(prev):])
        prev = cur
    return texts, starts, prev


def eligible_positions(texts: list[str], starts: list[int], full: str, limit: int | None = None) -> list[int]:
    """Eligible j: non-whitespace token whose first non-ws char is in CODE state, starting before `limit`."""
    states = lex_states(full)
    lim = len(full) if limit is None else limit
    out = []
    for j, (t, s) in enumerate(zip(texts, starts)):
        if not t.strip():
            continue
        off = s + (len(t) - len(t.lstrip()))
        if off >= lim or off >= len(states):
            continue
        if states[off] == CODE:
            out.append(j)
    return out


def entropy(top: dict[int, float]) -> float:
    """H = -sum p log p over the renormalised top-20."""
    lp = np.array(list(top.values()), dtype=np.float64)
    p = np.exp(lp - lp.max())
    p /= p.sum()
    return float(-(p * np.log(np.clip(p, 1e-300, None))).sum())


def column_at(full: str, offset: int) -> int:
    nl = full.rfind("\n", 0, offset)
    return offset - (nl + 1)


def candidate_ok(tok, tid: int, prefix_col: int) -> tuple[bool, str]:
    if tid >= SPECIAL_MIN_ID or tid in set(tok.all_special_ids):
        return False, "special"
    t = tok.decode([tid], skip_special_tokens=False, clean_up_tokenization_spaces=False)
    if "`" in t:
        return False, "backtick"
    if not t.strip():
        return False, "whitespace"
    for k in range(len(t) - 1):
        if t[k] == "\n" and t[k + 1] not in " \t\n":
            return False, "dedent_col0"
    if prefix_col == 0 and t[0] not in " \t\n":
        # 'def ' / 'class ' at column 0 (the plan's rule); any other non-space character at column 0 would also
        # close the function body (generalisation, logged in the deviations).
        return False, "def_col0" if t.startswith(("def ", "class ")) else "dedent_col0"
    return True, "ok"


def ranked(top: dict[int, float]) -> list[tuple[int, float]]:
    return sorted(top.items(), key=lambda kv: (-kv[1], kv[0]))[:20]


def candidates_at(tok, top: dict[int, float], col: int) -> list[tuple[int, int, float]]:
    """(rank, id, logprob) for ranks 2..8 (1-based) that pass the filters."""
    out = []
    for r, (tid, lp) in enumerate(ranked(top), start=1):
        if 2 <= r <= 8 and candidate_ok(tok, tid, col)[0]:
            out.append((r, tid, lp))
    return out


def lexical_class(t: str) -> str:
    s = t.strip()
    if not s:
        return "whitespace"
    if s.startswith("#"):
        return "comment-opener"
    low = s.lower()
    if s[0] in "'\"" or (len(s) > 1 and low[0] in "rbfu" and s[1] in "'\"") or (
            len(s) > 2 and low[:2] in ("rb", "br", "fr", "rf") and s[2] in "'\""):
        return "string-opener"
    if s[0].isdigit() or (s[0] == "." and len(s) > 1 and s[1].isdigit()):
        return "number"
    if s[0].isalpha() or s[0] == "_":
        word = ""
        for ch in s:
            if ch.isalnum() or ch == "_":
                word += ch
            else:
                break
        return "keyword" if keyword.iskeyword(word) else "identifier"
    return "operator"


def choose_branch(tok, body_ids: list[int], tops: list[dict[int, float]], text_limit: int, mode: str,
                  seed: int, arm: str, call: int) -> dict:
    """Pick (j*, forced id) for DTB ('entropy') or RPB ('random'); returns the full log record."""
    texts, starts, full = token_offsets(tok, body_ids)
    m = len(body_ids)
    elig = [j for j in eligible_positions(texts, starts, full, text_limit) if j < len(tops) and tops[j]]
    H = [entropy(tops[j]) if j < len(tops) and tops[j] else float("nan") for j in range(m)]
    rec = {"m": m, "eligible": elig, "H": [round(h, 6) if math.isfinite(h) else None for h in H],
           "no_branch": False, "j_star": None, "fallback_steps": 0}
    if len(elig) < 2:
        rec["no_branch"] = True
        rec["no_branch_reason"] = "fewer than 2 eligible positions"
        return rec
    rng_pos = np.random.default_rng([seed, ARM_CODE[arm], call, 4])
    order = sorted(elig, key=lambda j: (-H[j], j)) if mode == "entropy" else None
    tries = 0
    chosen, cands = None, []
    while True:
        if mode == "entropy":
            if tries >= len(order):
                break
            j = order[tries]
        else:
            if tries >= 11:  # first draw + up to 10 redraws
                break
            j = int(elig[int(rng_pos.integers(len(elig)))])
        col = column_at(full, starts[j])
        cands = candidates_at(tok, tops[j], col)
        if cands:
            chosen = j
            break
        tries += 1
    rec["fallback_steps"] = tries
    if chosen is None:
        rec["no_branch"] = True
        rec["no_branch_reason"] = "no candidate at any tried position"
        return rec
    j = chosen
    rng_tok = np.random.default_rng([seed, ARM_CODE[arm], call, 3])
    r, tid, lp = cands[int(rng_tok.integers(len(cands)))]
    sorted_elig_H = sorted((H[e] for e in elig), reverse=True)
    forced_text = tok.decode([tid], skip_special_tokens=False, clean_up_tokenization_spaces=False)
    rec.update({
        "j_star": j, "H_j_star": H[j], "H_rank_among_eligible": 1 + sorted_elig_H.index(H[j]),
        "n_eligible": len(elig), "top20_at_j": [[int(a), float(b)] for a, b in ranked(tops[j])],
        "sampled_id": int(body_ids[j]), "sampled_text": texts[j], "forced_id": int(tid), "forced_text": forced_text,
        "forced_rank": int(r), "forced_logprob": float(lp), "forced_eq_sampled": int(tid) == int(body_ids[j]),
        "forced_class": lexical_class(forced_text), "sampled_class": lexical_class(texts[j]),
        "candidates": [[int(a), int(b), float(c)] for a, b, c in cands], "position_frac": j / max(1, m),
    })
    return rec

### 3.1 Re-deriving the logged forcings
This repeats the check in `tests/test_rederive_forcing.py`. For every logged DTB/RPB call, it recomputes j\*, the
candidate set and the forced token from the logged request-1 token ids, log probs and seeds, and compares them with
the log. It then rebuilds the final generation as `decode(body[:j*] + forced token) + request-2 continuation`.

In [ ]:
dtb_recs = [e for e in EXAMPLES if e.get("r1") and e["arm"] == "DTB"]
rpb_recs = [e for e in EXAMPLES if e.get("r1") and e["arm"] == "RPB"]
forcing_recs = dtb_recs[:(N_FORCINGS + 1) // 2] + rpb_recs[:N_FORCINGS // 2]

n_match = n_text = n_branched = 0
print(f"{'arm':4s} {'seed':>4s} {'call':>5s} {'j*':>3s} {'H(j*)':>6s} {'H rank':>6s} {'rank':>4s}  "
      f"{'sampled token':>16s} -> {'forced token':16s} {'class':11s} same as log")
for e in forcing_recs:
    r1 = e["r1"]
    top = [{int(k): float(v) for k, v in d.items()} for d in r1["top"]]
    br = choose_branch(tok, r1["token_ids"], top, len(r1["text"]), "entropy" if e["arm"] == "DTB" else "random",
                       e["seed"], e["arm"], e["call_idx"])
    lg = e["branch_logged"]
    same = (br["no_branch"] == lg["no_branch"] and br.get("j_star") == lg.get("j_star") and
            br.get("forced_id") == lg.get("forced_id") and br.get("candidates") == lg.get("candidates"))
    n_match += same
    e["branch_rederived"] = br
    if br["no_branch"]:
        print(f"{e['arm']:4s} {e['seed']:4d} {e['call_idx']:5d}  no branch ({br.get('no_branch_reason')})  {same}")
        continue
    # the generation the evaluator saw = decoded forced prefix + the logged request-2 continuation
    n_branched += 1
    pre = tok.decode(r1["token_ids"][:br["j_star"]] + [br["forced_id"]], skip_special_tokens=False,
                     clean_up_tokenization_spaces=False)
    n_text += (pre + (e["r2_text"] or "")) == e["final_text"]
    print(f"{e['arm']:4s} {e['seed']:4d} {e['call_idx']:5d} {br['j_star']:3d} {br['H_j_star']:6.3f} "
          f"{br['H_rank_among_eligible']:3d}/{br['n_eligible']:<2d} {br['forced_rank']:4d}  {br['sampled_text']!r:>16s} -> "
          f"{br['forced_text']!r:16s} {br['forced_class']:11s} {same}")
print(f"\nre-derived forcings identical to the log (j*, candidate set, forced token): {n_match}/{len(forcing_recs)}")
print(f"final generation = forced prefix + logged request-2 continuation: {n_text}/{n_branched}")

### 3.2 Where DTB and RPB branch
The plot shows the entropy at each of the 48 request-1 positions for one DTB call and one RPB call. Blue bars are
eligible code positions, grey bars are comments, strings or whitespace, and the red bar is j\*. DTB branches at the
model's most uncertain code token. RPB usually lands on a near-deterministic token, so its forced alternative is an
unlikely detour rather than a fork the model was already weighing.

In [ ]:
show = [x for x in (next((e for e in forcing_recs if e["arm"] == "DTB"), None),
                     next((e for e in forcing_recs if e["arm"] == "RPB"), None)) if x is not None]
fig, axes = plt.subplots(len(show), 1, figsize=(15, 3.6 * len(show)), squeeze=False)
for ax, e in zip(axes[:, 0], show):
    br = e["branch_rederived"]
    texts, _, _ = token_offsets(tok, e["r1"]["token_ids"])
    H = np.array([h if h is not None else np.nan for h in br["H"]])
    elig = set(br["eligible"])
    colors = ["tab:red" if j == br["j_star"] else ("tab:blue" if j in elig else "lightgray") for j in range(br["m"])]
    ax.bar(range(br["m"]), H, color=colors)
    ax.set_xticks(range(br["m"]))
    ax.set_xticklabels([t.replace("\n", "\\n") for t in texts], rotation=90, fontsize=7, family="monospace")
    ax.set_xlim(-0.6, br["m"] - 0.4)
    ax.set_ylabel("entropy of top-20 (nats)")
    if br["no_branch"]:
        ax.set_title(f"{e['arm']}|{e['seed']} call {e['call_idx']}: no branch", fontsize=10)
    else:
        ax.set_title(f"{e['arm']}|{e['seed']} call {e['call_idx']}: j* = {br['j_star']} (H = {br['H_j_star']:.2f} nats, "
                     f"rank {br['H_rank_among_eligible']}/{br['n_eligible']} among eligible) - sampled "
                     f"{br['sampled_text']!r}, forced rank-{br['forced_rank']} {br['forced_text']!r}", fontsize=10)
plt.tight_layout()
plt.show()

## 4. Search, step 3: the FunSearch programs database (`src/islands.py`)
Each (arm, seed) search keeps 5 islands seeded with NN, and programs on an island are grouped into clusters by their 64
per-instance gaps. To build a prompt, the database:
1. draws an island uniformly;
2. draws up to 2 clusters with probability `softmax(score / T_c)` under FunSearch's temperature schedule;
3. within each cluster, prefers shorter programs.

Every 400 calls the 2 worst islands are reset. All randomness comes from per-(arm, seed) RNG streams. The cell below
replays the island-choice stream of each arm-seed and checks it against the island logged for every call in the data.

In [ ]:
# ---- src/islands.py (FunSearch-style programs database) ----
N_ISLANDS = 5
FUNCS_PER_PROMPT = 2
T_INIT = 0.1
T_PERIOD = 30_000
RESET_EVERY = 400
N_RESET = 2


def _softmax(logits: np.ndarray, temperature: float) -> np.ndarray:
    logits = np.asarray(logits, dtype=np.float64)
    if not np.all(np.isfinite(logits)):
        raise ValueError(f"non-finite logits {logits}")
    result = scipy.special.softmax(logits / temperature, axis=-1)
    index = int(np.argmax(result))
    result[index] = 1 - np.sum(result[0:index]) - np.sum(result[index + 1:])
    return result


@dataclass
class Prog:
    pid: str            # call id "arm|seed|call" or "NN"
    body: str
    score: float        # -F
    length: int         # characters of the body


@dataclass
class Cluster:
    score: float
    programs: list[Prog] = field(default_factory=list)

    def sample(self, rng: np.random.Generator) -> Prog:
        lengths = np.array([p.length for p in self.programs], dtype=np.float64)
        normalized = (lengths - lengths.min()) / (lengths.max() + 1e-6)
        probs = _softmax(-normalized, 1.0)
        return self.programs[int(rng.choice(len(self.programs), p=probs))]


@dataclass
class Island:
    clusters: dict = field(default_factory=dict)  # signature -> Cluster
    num_programs: int = 0

    def register(self, prog: Prog, signature: tuple) -> None:
        if signature not in self.clusters:
            self.clusters[signature] = Cluster(score=prog.score)
        self.clusters[signature].programs.append(prog)
        self.num_programs += 1

    def get_prompt(self, rng: np.random.Generator) -> list[Prog]:
        sigs = list(self.clusters.keys())
        scores = np.array([self.clusters[s].score for s in sigs])
        temperature = T_INIT * (1 - (self.num_programs % T_PERIOD) / T_PERIOD)
        probs = _softmax(scores, temperature)
        k = min(len(self.clusters), FUNCS_PER_PROMPT)
        idx = rng.choice(len(sigs), size=k, p=probs)
        chosen = [self.clusters[sigs[i]] for i in idx]
        impl = [c.sample(rng) for c in chosen]
        sc = [c.score for c in chosen]
        order = np.argsort(sc, kind="stable")
        return [impl[i] for i in order]


class Database:
    def __init__(self, arm: str, seed: int, nn: Prog, nn_signature: tuple):
        self.arm, self.seed = arm, seed
        a = ARM_CODE[arm]
        self.rng_island = np.random.default_rng([seed, a, 1])
        self.rng_reset = np.random.default_rng([seed, a, 5])
        self.rng_sample = np.random.default_rng([seed, a, 6])
        self.islands = [Island() for _ in range(N_ISLANDS)]
        self.best_score = [-np.inf] * N_ISLANDS
        self.best_prog: list[Prog | None] = [None] * N_ISLANDS
        self.best_sig: list[tuple | None] = [None] * N_ISLANDS
        self.completed_calls = 0
        self.resets: list[dict] = []
        for i in range(N_ISLANDS):
            self._register_in_island(nn, i, nn_signature)

    def _register_in_island(self, prog: Prog, island: int, signature: tuple) -> None:
        self.islands[island].register(prog, signature)
        if prog.score > self.best_score[island]:
            self.best_score[island] = prog.score
            self.best_prog[island] = prog
            self.best_sig[island] = signature

    def get_prompt(self) -> tuple[int, list[Prog]]:
        island = int(self.rng_island.integers(N_ISLANDS))
        return island, self.islands[island].get_prompt(self.rng_sample)

    def complete_call(self, island: int, prog: Prog | None, signature: tuple | None) -> None:
        """Register one finished call in call order (invalid offspring are discarded but count as calls)."""
        if prog is not None:
            self._register_in_island(prog, island, signature)
        self.completed_calls += 1
        if self.completed_calls % RESET_EVERY == 0:
            self.reset_islands()

    def reset_islands(self) -> None:
        noisy = np.array(self.best_score) + self.rng_reset.standard_normal(N_ISLANDS) * 1e-6
        order = np.argsort(noisy, kind="stable")
        reset_ids, keep_ids = order[:N_RESET], order[N_RESET:]
        log = {"at_call": self.completed_calls, "reset": [int(i) for i in reset_ids], "seeded_from": []}
        for i in reset_ids:
            self.islands[int(i)] = Island()
            self.best_score[int(i)] = -np.inf
            self.best_prog[int(i)] = None
            founder = int(self.rng_reset.choice(keep_ids))
            log["seeded_from"].append(founder)
            self._register_in_island(self.best_prog[founder], int(i), self.best_sig[founder])
        self.resets.append(log)

    def snapshot(self) -> dict:
        return {"arm": self.arm, "seed": self.seed, "completed_calls": self.completed_calls, "resets": self.resets,
                "best_score": [float(s) for s in self.best_score],
                "best_pid": [p.pid if p else None for p in self.best_prog],
                "islands": [{"num_programs": isl.num_programs, "n_clusters": len(isl.clusters),
                             "clusters": [{"score": c.score, "pids": [p.pid for p in c.programs]}
                                          for c in isl.clusters.values()]} for isl in self.islands]}

    def snapshot_json(self) -> str:
        return json.dumps(self.snapshot(), sort_keys=True)


# ---- replay: one island draw per call, in call order (src/search.py builds every prompt via Database.get_prompt) ----
nn_prog = Prog(pid="NN", body=nn_body(NN_PROGRAM), score=-1.0, length=len(nn_body(NN_PROGRAM)))
nn_sig = tuple(round(float(g), 6) for g in nn_sel_gaps)
n_ok = n_all = 0
for arm, seed in sorted({(e["arm"], e["seed"]) for e in EXAMPLES if e["arm"] != "NOSEL"}):
    logged_island = {e["call_idx"]: e["island"] for e in EXAMPLES if (e["arm"], e["seed"]) == (arm, seed)}
    db = Database(arm, seed, nn_prog, nn_sig)
    drawn = [db.get_prompt()[0] for _ in range(max(logged_island) + 1)]
    ok = sum(drawn[c] == isl for c, isl in logged_island.items())
    n_ok, n_all = n_ok + ok, n_all + len(logged_island)
    print(f"{arm:5s}|{seed}: {ok}/{len(logged_island)} logged islands reproduced "
          f"(calls {min(logged_island)}..{max(logged_island)})")
print(f"island choice replayed from RNG stream [seed, arm, 1]: {n_ok}/{n_all} identical to the log")

## 5. Search, step 4: replaying logged calls through the evaluation path (`src/search.py`)
`Search.evaluate` turns one finished generation into one row of `data/calls.jsonl`:
1. extract the program;
2. apply the static check and compute the AST hash;
3. execute the program in the sandbox (64 SELECTION instances + 2,000 probes);
4. compute gaps and F, then label the program **valid** / **probe-valid** (≤ 10% ERR decisions) or name its error.

The function is copied below as a plain function. Two changes: no `self` (the NN normalisation, references and caches
are notebook variables), and a direct, cached call to `evaluate_program` instead of the async pool of evaluator processes.

In [ ]:
# ---- src/search.py (excerpts: sampling settings, request seeds, Search.evaluate) ----
ARM_SAMPLING = {"T1.0": (1.0, 1.0), "T1.6": (1.6, 0.95), "IDEA": (1.0, 1.0), "NOSEL": (1.0, 1.0),
                "DTB": (1.0, 1.0), "RPB": (1.0, 1.0)}
ARM_ORDER = ["T1.0", "T1.6", "DTB", "RPB", "IDEA", "NOSEL"]
ENGINE_TAG = "vllm0.11.0|Qwen3-8B-FP8@220b46e|marlin|O3-piecewise+fulldecode"


def req_seed(seed: int, arm: str, call: int) -> int:
    return int(np.random.default_rng([seed, ARM_CODE[arm], call, 2]).integers(2 ** 31))


def utc_now() -> str:   # src/common.py
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())


EVAL_CACHE: dict[str, dict] = {}       # Search.cache: ast_hash -> raw evaluation
DECISIONS: dict[str, list[int]] = {}   # ProbeStore (data/probe_decisions.bin): ast_hash -> 2,000 decisions


def _eval_code(code: str, h: str) -> dict:
    """Search._eval_code without the async evaluator pool: one evaluation per AST hash, decisions kept."""
    if h in EVAL_CACHE:
        return EVAL_CACHE[h]
    res = evaluate_program(code, True)
    if res.get("decisions") is not None:
        DECISIONS[h] = res["decisions"]
    raw = {k: v for k, v in res.items() if k != "decisions"}
    EVAL_CACHE[h] = raw
    return raw


def evaluate(rec: dict) -> dict:
    """Search.evaluate: one finished generation (request record) -> one calls.jsonl row."""
    arm, seed, c = rec["arm"], rec["seed"], rec["call_idx"]
    k = rec["k"]
    row = {"model": "Qwen/Qwen3-8B-FP8@220b46e", "engine": ENGINE_TAG, "task": "tsp_construct", "arm": arm,
           "seed": seed, "call_idx": c, "round": rec["round"], "island": rec["island"],
           "parent_ids": rec["parent_ids"], "parent_F": rec["parent_F"], "k": k,
           "prompt_tokens": rec.get("prompt_tokens"), "gen_tokens": rec.get("gen_tokens"),
           "discarded_tokens": rec.get("discarded_tokens", 0), "gen_wall": rec.get("gen_wall"),
           "request_seed": rec["request_seed"], "truncated": rec.get("final_finish") == "length",
           "label": None, "probe_valid": False, "F": None, "G50": None, "G100": None, "gaps": None,
           "code": None, "body": None, "code_sha256": None, "ast_hash": None, "trimmed_lines": 0,
           "eval_raw": None, "utc": utc_now()}
    if arm in ("DTB", "RPB") and rec.get("branch"):
        br = rec["branch"]
        row["dtb"] = {kk: br.get(kk) for kk in (
            "m", "eligible", "H", "j_star", "H_j_star", "H_rank_among_eligible", "n_eligible", "top20_at_j",
            "sampled_id", "sampled_text", "forced_id", "forced_text", "forced_rank", "forced_logprob",
            "forced_eq_sampled", "forced_class", "sampled_class", "no_branch", "no_branch_reason",
            "fallback_steps", "candidates", "position_frac")}
        row["dtb"]["r1_tokens"] = (rec.get("r1") or {}).get("completion_tokens")
        row["dtb"]["r2_tokens"] = (rec.get("r2") or {}).get("completion_tokens")
        row["dtb"]["r1_finish"] = (rec.get("r1") or {}).get("finish_reason")
    if rec.get("final_text") is None:
        row["label"] = "gen_failed"
        row["gen_error"] = rec.get("gen_error")
        return row
    ex = extract(rec["final_text"], k)
    row["trimmed_lines"] = ex["trimmed_lines"]
    row["body"] = ex["body"]
    if ex["program"] is None:
        row["label"] = "truncated" if row["truncated"] else ex["label"]
        return row
    code = ex["program"]
    row["code"] = code
    row["code_sha256"] = sha256_text(code)
    try:
        san = static_check(code)
        h = ast_hash(san)
    except StaticReject as e:
        row["label"] = e.cls
        row["reason"] = e.reason
        row["ast_hash"] = ast_hash(code)
        return row
    row["ast_hash"] = h
    raw = _eval_code(code, h)
    row["eval_raw"] = raw
    row["label"] = raw["label"]
    if raw["label"] == "valid":
        gaps = gaps_from_lengths(raw["lengths"], refs["selection"])
        F, g50, g100 = fitness(gaps, NN_G50, NN_G100)
        row.update(F=F, G50=g50, G100=g100, gaps=[round(float(g), 6) for g in gaps],
                   probe_valid=raw.get("probe_label") == "probe_valid", probe_label=raw.get("probe_label"),
                   n_err=raw.get("n_err"))
    else:
        row["reason"] = raw.get("reason")
        if row["truncated"] and raw["label"] in ("syntax_error", "no_code"):
            row["label"] = "truncated"
    return row


def request_record(e: dict) -> dict:
    """Notebook helper: the logged request record (data/requests/<arm>_<seed>.jsonl) of a call, as Search.evaluate
    receives it after generation; for DTB/RPB the branch is the one re-derived in section 3 when available."""
    rec = {k: e[k] for k in ("arm", "seed", "call_idx", "round", "island", "k", "parent_ids", "parent_F",
                             "request_seed", "prompt_tokens", "gen_tokens", "discarded_tokens", "final_text",
                             "final_finish")}
    if e.get("r1"):
        rec["r1"] = e["r1"]
        rec["branch"] = e.get("branch_rederived") or e["branch_logged"]
    return rec

### 5.1 Run the replay
The next cell replays two groups of logged calls:
- `N_CALLS_PER_ARM` **matched calls** of every arm (round 6, seed 0, the same call indices in every arm);
- `N_BEST_PROGRAMS` **arm-seed best programs**.

Each replayed row is compared with the logged one: label, request seed, code hash, AST hash, F, and the full
2,000-decision behaviour vector. Instances, probe states and the RNG seeds inside the sandbox are all fixed, so a
faithful re-execution must match the log exactly.

In [ ]:
block = [e for e in EXAMPLES if e["group"] == "block" and e["call_idx"] < 192 + N_CALLS_PER_ARM]
bests = sorted((e for e in EXAMPLES if e["group"] == "best"),
               key=lambda e: (e["arm"] == "NOSEL", e["seed"], ARM_ORDER.index(e["arm"])))[:N_BEST_PROGRAMS]


def decode_hex(h: str) -> list[int]:
    """A logged decision vector: one byte per probe state, 255 = ERR (-1)."""
    return [b if b != 255 else ERR for b in bytes.fromhex(h)]


def _same(a, b) -> bool:
    return (a is None and b is None) or (a is not None and b is not None and abs(a - b) <= 1e-9 * max(1.0, abs(b)))


REPLAYED = []   # (logged example, replayed row, wall seconds)
t0 = time.time()
for e in block + bests:
    t1 = time.time()
    row = evaluate(request_record(e))
    REPLAYED.append((e, row, time.time() - t1))
print(f"replayed {len(REPLAYED)} logged calls through Search.evaluate in {time.time() - t0:.1f}s\n")

agg = defaultdict(Counter)
for e, row, wall in REPLAYED:
    a = agg[(e["group"], e["arm"])]
    a["calls"] += 1
    a["valid"] += row["label"] == "valid"
    a["probe_valid"] += bool(row["probe_valid"])
    a["label"] += row["label"] == e["label"]
    a["seed"] += req_seed(e["seed"], e["arm"], e["call_idx"]) == e["request_seed"]
    a["code"] += row["code_sha256"] == e["code_sha256"]
    a["ast"] += row["ast_hash"] == e["ast_hash"]
    a["F"] += _same(row["F"], e["F"])
    if e["decisions_hex"]:
        a["dec_logged"] += 1
        a["dec"] += DECISIONS.get(row["ast_hash"]) == decode_hex(e["decisions_hex"])
    a["wall"] += wall
hdr = (f"{'group':6s} {'arm':5s} {'calls':>5s} {'valid':>5s} {'p-valid':>7s} | same as log: {'label':>6s} "
       f"{'req.seed':>8s} {'code':>6s} {'AST':>6s} {'F':>6s} {'2000 decisions':>14s} | {'wall s':>6s}")
print(hdr)
print("-" * len(hdr))
for (g, arm), a in agg.items():
    print(f"{g:6s} {arm:5s} {a['calls']:5d} {a['valid']:5d} {a['probe_valid']:7d} | {'':12s} "
          f"{a['label']:3d}/{a['calls']:<2d} {a['seed']:5d}/{a['calls']:<2d} {a['code']:3d}/{a['calls']:<2d} "
          f"{a['ast']:3d}/{a['calls']:<2d} {a['F']:3d}/{a['calls']:<2d} {a['dec']:11d}/{a['dec_logged']:<2d} | "
          f"{a['wall']:6.1f}")
tot = sum(agg.values(), Counter())
print("-" * len(hdr))
print(f"all: labels {tot['label']}/{tot['calls']}, AST hashes {tot['ast']}/{tot['calls']}, F {tot['F']}/{tot['calls']}, "
      f"behaviour vectors {tot['dec']}/{tot['dec_logged']} identical to the logged run")

e_best, r_best, _ = min((x for x in REPLAYED if x[1]["F"] is not None), key=lambda x: x[1]["F"])
print(f"\nbest replayed program: {e_best['arm']}|{e_best['seed']} call {e_best['call_idx']}  F = {r_best['F']:.4f} "
      f"(NN = 1), G50 = {r_best['G50']:.2f}%, G100 = {r_best['G100']:.2f}% (NN: {NN_G50:.2f}%, {NN_G100:.2f}%)")
print(r_best["code"])

## 6. Analysis: behaviour clusters (`src/analysis.py`)
These are the pre-registered statistics, frozen in COMMIT #1 before the first search call:
- Identical decision vectors are collapsed first.
- The unique vectors are clustered by **complete linkage** under Hamming distance, cut at 0.05, so every cluster has a
  minimum within-cluster agreement ≥ 0.95.
- The primary unit of V2-T is **clusters per 1,000 calls**. Invalid calls count in the denominator, so an arm that
  wastes calls on broken programs is penalised.

In [ ]:
# ---- src/analysis.py (pre-registered analysis; the functions used by this demo) ----
CORE_ARMS = ["T1.0", "T1.6", "DTB", "RPB"]
TEMP_ARMS = ["T1.0", "T1.6"]
SEEDS = [0, 1]
THRESH = 0.05


# ------------------------------------------------------------------ clustering
def unique_vectors(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """(unique rows U, inverse index per row, multiplicity per unique row)."""
    if len(X) == 0:
        return X.reshape(0, X.shape[1] if X.ndim == 2 else 0), np.zeros(0, int), np.zeros(0, int)
    U, inv, cnt = np.unique(X, axis=0, return_inverse=True, return_counts=True)
    return U, inv.reshape(-1), cnt


def cluster_labels_unique(U: np.ndarray, t: float = THRESH) -> np.ndarray:
    """Complete-linkage flat clusters of unique decision vectors (labels 1..K)."""
    if len(U) == 0:
        return np.zeros(0, int)
    if len(U) == 1:
        return np.ones(1, int)
    Z = linkage(pdist(U, "hamming"), method="complete")
    return fcluster(Z, t=t, criterion="distance")


def cluster_programs(X: np.ndarray, t: float = THRESH) -> dict:
    """Cluster the rows of X (one decision vector per program). Returns per-program labels and the count."""
    U, inv, cnt = unique_vectors(X)
    lab_u = cluster_labels_unique(U, t)
    labels = lab_u[inv] if len(inv) else np.zeros(0, int)
    return {"n_programs": int(len(X)), "n_unique": int(len(U)), "n_clusters": int(len(set(lab_u.tolist()))),
            "labels": labels, "labels_unique": lab_u, "U": U, "mult": cnt}


def min_within_agreement(U: np.ndarray, lab: np.ndarray) -> float:
    """Exact property check: the minimum pairwise agreement inside every cluster (1.0 if all singletons)."""
    worst = 1.0
    for c in set(lab.tolist()):
        idx = np.where(lab == c)[0]
        if len(idx) < 2:
            continue
        d = pdist(U[idx], "hamming")
        worst = min(worst, 1.0 - float(d.max()))
    return worst


def shannon(labels: np.ndarray) -> float:
    if len(labels) == 0:
        return 0.0
    c = np.array(list(Counter(labels.tolist()).values()), dtype=float)
    p = c / c.sum()
    return float(-(p * np.log(p)).sum())


def discovery_curve(call_idx: np.ndarray, labels: np.ndarray, n_calls: int) -> list[int]:
    """Distinct clusters among probe-valid programs with call index <= c, for c = 0..n_calls-1."""
    order = np.argsort(call_idx, kind="stable")
    seen, out, k = set(), [], 0
    ci = call_idx[order]
    lab = labels[order]
    for c in range(n_calls):
        while k < len(ci) and ci[k] <= c:
            seen.add(int(lab[k]))
            k += 1
        out.append(len(seen))
    return out


# ------------------------------------------------------------------ ratios and bootstrap
def pooled_ratio(c: dict, N: dict, a: str, b: str, seeds=SEEDS) -> float:
    num = sum(c[(a, s)] for s in seeds) / max(1, sum(N[(a, s)] for s in seeds))
    den = sum(c[(b, s)] for s in seeds) / max(1, sum(N[(b, s)] for s in seeds))
    return float(num / den) if den > 0 else float("inf")


def per_seed_ratio(c: dict, N: dict, a: str, b: str, s: int) -> float:
    den = c[(b, s)] / max(1, N[(b, s)])
    return float((c[(a, s)] / max(1, N[(a, s)])) / den) if den > 0 else float("inf")


def paired_bootstrap_diff(a: np.ndarray, b: np.ndarray, n_boot: int = 10000, seed: int = 0) -> dict:
    """Paired per-instance bootstrap of mean(a - b) (gap points)."""
    d = np.asarray(a, float) - np.asarray(b, float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), (n_boot, len(d)))
    m = d[idx].mean(axis=1)
    return {"mean_diff": float(d.mean()), "lo": float(np.percentile(m, 2.5)), "hi": float(np.percentile(m, 97.5)),
            "n": int(len(d))}


# ------------------------------------------------------------------ verdicts
def verdict_v1(g200: dict, g200_nn: float, arms=CORE_ARMS, seeds=SEEDS) -> dict:
    """g200[(arm, seed)] = held-out n-200 mean gap of that arm-seed's selection-best program."""
    bar = 0.8 * g200_nn
    keys = [(a, s) for a in arms for s in seeds if (a, s) in g200 and g200[(a, s)] is not None]
    missing = [(a, s) for a in arms for s in seeds if (a, s) not in keys]
    failing = [f"{a}|{s}" for a, s in keys if not g200[(a, s)] <= bar]
    if not missing and not failing:
        v = "SUPPORTED"
    elif not any(g200[k] < g200_nn for k in keys):
        v = "DISCONFIRMED"
    else:
        v = "PARTIAL"
    return {"verdict": v, "bar_0.8xNN": bar, "G200_NN": g200_nn, "failing": failing,
            "missing": [f"{a}|{s}" for a, s in missing],
            "per_arm_seed": {f"{a}|{s}": g200[(a, s)] for a, s in keys}}


def verdict_v2(c: dict, N: dict, g200: dict, seeds=SEEDS) -> dict:
    """c[(arm, seed)] primary cluster counts, N[(arm, seed)] calls, g200[(arm, seed)] held-out best."""
    R = {x: pooled_ratio(c, N, "DTB", x, seeds) for x in ("T1.0", "T1.6", "RPB")}
    r = {f"{x}|{s}": per_seed_ratio(c, N, "DTB", x, s) for x in ("T1.0", "T1.6", "RPB") for s in seeds}

    def mean_g(arm):
        v = [g200.get((arm, s)) for s in seeds]
        return float(np.mean(v)) if all(x is not None for x in v) else None

    gd, gt = mean_g("DTB"), [mean_g(x) for x in TEMP_ARMS]
    heldout_ok = gd is not None and all(x is not None for x in gt) and gd <= min(gt)
    c1 = R["T1.0"] >= 1.5 and R["T1.6"] >= 1.5
    c2 = all(r[f"{x}|{s}"] > 1 for x in TEMP_ARMS for s in seeds)
    c3 = R["RPB"] >= 1.2
    if c1 and c2 and c3 and heldout_ok:
        v, branches = "SUPPORTED", []
    elif R["T1.0"] <= 1.0 or R["T1.6"] <= 1.0:
        v, branches = "DISCONFIRMED", []
    else:
        v, branches = "PARTIAL", []
        if not heldout_ok:
            branches.append("clusters but worse held-out best")
        if sum(R[x] >= 1.5 for x in TEMP_ARMS) == 1:
            branches.append("one comparator only")
        if not c2 and any(all(r[f"{x}|{s}"] > 1 for x in TEMP_ARMS) for s in seeds):
            branches.append("one seed only")
        if not c3:
            branches.append("not position-specific")
        if not branches:
            branches.append("ratios between 1.0 and 1.5")
    return {"verdict": v, "branches": branches, "R": R, "r": r, "criteria": {
        "R_temp_ge_1.5": c1, "per_seed_gt_1": c2, "R_RPB_ge_1.2": c3, "heldout_DTB_le_temp": heldout_ok},
        "mean_G200": {"DTB": gd, "T1.0": gt[0], "T1.6": gt[1]}}


def best_program(rows: list[dict]) -> dict | None:
    """Minimum F among PROBE-VALID programs (ties to the earliest call), chosen without held-out data."""
    pv = [r for r in rows if r.get("probe_valid") and r.get("F") is not None]
    if not pv:
        return None
    return min(pv, key=lambda r: (r["F"], r["call_idx"]))

### 6.1 Clusters among the replayed matched calls, and the geometry of behaviour space
The cluster rule is applied to the **replayed** behaviour vectors of each arm's matched calls, following the per-arm
loop of `src/analyze_run.py`. With at most 16 calls per arm this is a small sample (the run used 1,056 per arm-seed), so
the logged full-run rates are printed alongside. Within a 16-call window almost every valid program is its own
cluster, so these windowed rates sit far above the full-run rates, which count clusters cumulatively over 1,056
calls. Compare arms with each other here, not with the full run.

Next, all replayed probe-valid programs plus NN are pooled, and their pairwise Hamming distances (1 − agreement on the
2,000 probe decisions) are embedded with **classical MDS**. The embedding is plotted in section 9: each point is a
heuristic, and close points make nearly the same decisions.

In [ ]:
ARMS = ["T1.0", "T1.6", "DTB", "RPB", "IDEA"]   # src/analyze_run.py
FULL = CTX["full_run"]["per_arm_seed"]
block_rows = [(e, r) for e, r, _ in REPLAYED if e["group"] == "block"]

# ---- per-arm clusters on the replayed matched calls (the 5.2 loop of src/analyze_run.py, one seed) ----
demo_c, demo_N = {}, {}
print(f"{'arm':5s} {'calls':>5s} {'p-valid':>7s} {'unique':>6s} {'clusters':>8s} {'per 1k calls':>12s} "
      f"{'min agree':>9s} | full run per 1k calls: seed 0 / seed 1")
for arm in ARMS:
    rs = sorted([r for e, r in block_rows if e["arm"] == arm], key=lambda r: r["call_idx"])
    pv = [r for r in rs if r.get("probe_valid") and r.get("ast_hash") in DECISIONS]
    X = np.array([DECISIONS[r["ast_hash"]] for r in pv]) if pv else np.zeros((0, 2000), np.int16)
    res = cluster_programs(X)
    demo_c[(arm, 0)], demo_N[(arm, 0)] = res["n_clusters"], len(rs)
    mwa = min_within_agreement(res["U"], res["labels_unique"]) if res["n_unique"] else float("nan")
    print(f"{arm:5s} {len(rs):5d} {len(pv):7d} {res['n_unique']:6d} {res['n_clusters']:8d} "
          f"{1000 * res['n_clusters'] / max(1, len(rs)):12.1f} {mwa:9.3f} | "
          f"{FULL[f'{arm}|0']['rate_per_1000_calls']:7.1f} / {FULL[f'{arm}|1']['rate_per_1000_calls']:.1f}")

# ---- pooled behaviour space: every replayed probe-valid program + NN ----
pool = [(e, r) for e, r, _ in REPLAYED if r.get("probe_valid") and r.get("ast_hash") in DECISIONS]
X_pool = np.array([DECISIONS[r["ast_hash"]] for _, r in pool] + [nn_res["decisions"]], dtype=np.int16)
pool_meta = [(e["group"], e["arm"], e["seed"], e["call_idx"], r["F"]) for e, r in pool] + [("nn", "NN", 0, -1, 1.0)]
res_pool = cluster_programs(X_pool)
D_pool = squareform(pdist(X_pool, "hamming"))       # 1 - agreement on the 2,000 probe decisions
# classical (Torgerson) MDS of the Hamming distances
n_pool = len(D_pool)
J = np.eye(n_pool) - np.ones((n_pool, n_pool)) / n_pool
B = -0.5 * J @ (D_pool ** 2) @ J
w, V = np.linalg.eigh(B)
top2 = np.argsort(w)[::-1][:2]
MDS_XY = V[:, top2] * np.sqrt(np.maximum(w[top2], 0))
pos = np.maximum(w, 0).sum()
mds_share = float(np.maximum(w[top2], 0).sum() / pos) if pos > 0 else float("nan")
print(f"\npooled: {len(X_pool)} probe-valid programs (incl. NN) -> {res_pool['n_unique']} unique decision vectors -> "
      f"{res_pool['n_clusters']} behaviour clusters; 2-D MDS keeps {100 * mds_share:.0f}% of the positive spectrum")
agree_nn = [agreement(x, nn_res["decisions"]) for x in X_pool[:-1]]
if agree_nn:
    print(f"agreement with NN: median {np.median(agree_nn):.3f}, min {np.min(agree_nn):.3f} "
          f"(1.0 = identical decisions in all 2,000 probe states)")

## 7. The pre-registered verdicts, recomputed from the logged full-run numbers
The demo slice is far too small to test anything. The verdict rules, however, are the original functions, so the next
cell feeds them the **full-run** cluster counts (1,056 calls per arm-seed) and the held-out gaps of the 10 arm-seed best
programs, both stored in the demo data:
- **V2-T** (the claim under test) is SUPPORTED only if all of the following hold: DTB/T1.0 and DTB/T1.6 pooled ratios
  ≥ 1.5; every per-seed ratio > 1; DTB/RPB ≥ 1.2; DTB's held-out best no worse than the temperature arms'. It is
  **DISCONFIRMED** if DTB/T1.0 or DTB/T1.6 ≤ 1.0.
- **V1-T** is SUPPORTED if every core arm-seed best closes ≥ 20% of NN's held-out gap (G200 ≤ 0.8 × G200(NN)).

The cell also recomputes the paired per-instance bootstrap of the DTB-minus-others held-out gap and applies the same
ratio to the replayed matched calls (illustration only).

In [ ]:
def _key(k: str) -> tuple[str, int]:
    a, s = k.split("|")
    return a, int(s)


c_full = {_key(k): v["primary_clusters"] for k, v in FULL.items()}
N_full = {_key(k): v["calls"] for k, v in FULL.items()}
g200_full = {_key(k): v["G200_best"] for k, v in FULL.items()}
G200_NN = CTX["full_run"]["G200_NN"]

v2 = verdict_v2(c_full, N_full, g200_full)
v1 = verdict_v1(g200_full, G200_NN)
logged_v1, logged_v2 = CTX["full_run"]["verdict_v1"], CTX["full_run"]["verdict_v2"]

print(f"V2-T (DTB finds more behaviour clusters per call than temperature): {v2['verdict']}   "
      f"[logged: {logged_v2['verdict']}]")
print("  pooled ratios R = DTB / X:", {k: round(x, 3) for k, x in v2["R"].items()})
print("  per-seed ratios r:        ", {k: round(x, 3) for k, x in v2["r"].items()})
print("  criteria:", v2["criteria"])
print("  mean held-out G200 of the bests:", {k: round(x, 2) for k, x in v2["mean_G200"].items()})
boot = CTX["full_run"]["ratios_per_seed"]
print("  logged round-block bootstrap 95% CIs of DTB/X:",
      {k: f"{v['ratio']:.2f} [{v['boot95']['lo']:.2f}, {v['boot95']['hi']:.2f}]" for k, v in boot.items()})
print(f"\nV1-T (every core arm-seed best closes >= 20% of NN's held-out gap): {v1['verdict']}   "
      f"[logged: {logged_v1['verdict']}]")
print(f"  G200(NN) = {G200_NN:.2f}%, bar 0.8 x NN = {v1['bar_0.8xNN']:.2f}%")
print("  G200 of the arm-seed bests:", {k: round(x, 2) for k, x in v1["per_arm_seed"].items()})

print("\nPaired per-instance bootstrap, DTB best minus X best on the 32 held-out instances (gap points, 95% CI):")
for s in SEEDS:
    for xx in ("T1.0", "T1.6", "RPB", "IDEA"):
        d = paired_bootstrap_diff(np.array(FULL[f"DTB|{s}"]["heldout_gaps_best"]),
                                  np.array(FULL[f"{xx}|{s}"]["heldout_gaps_best"]))
        lg = CTX["full_run"]["paired_DTB_vs"][f"DTB-{xx}|{s}"]
        print(f"  DTB-{xx:4s}|{s}: {d['mean_diff']:+.2f} [{d['lo']:+.2f}, {d['hi']:+.2f}]   "
              f"logged {lg['mean_diff']:+.2f} [{lg['lo']:+.2f}, {lg['hi']:+.2f}]")

print(f"\nSame ratio on the replayed matched calls (seed 0, {N_CALLS_PER_ARM} calls per arm; illustration only):")
print("  DTB / X:", {x: round(pooled_ratio(demo_c, demo_N, "DTB", x, seeds=[0]), 3) for x in ("T1.0", "T1.6", "RPB", "IDEA")})

## 8. Held-out check (V1-T): best programs on unseen n = 200 instances
Each arm-seed's best program was chosen on SELECTION data only. Here NN and the first `N_HELDOUT_PROGRAMS` re-evaluated
best programs are re-executed through `evaluate_set` on the first `N_HELDOUT` held-out instances (n = 200, never seen
by the search). Their gaps to LKH-3 are compared with the logged per-instance gaps.

In [ ]:
held_progs = [("NN", NN_PROGRAM, np.array(CTX["nn_baseline"]["heldout_gaps"]))]
for e, r, _ in REPLAYED:
    if e["group"] == "best" and r["code"] and len(held_progs) <= N_HELDOUT_PROGRAMS:
        key = f"{e['arm']}|{e['seed']}"
        logged = FULL[key]["heldout_gaps_best"] if key in FULL else CTX["full_run"]["NOSEL|0"]["heldout_gaps_best"]
        held_progs.append((key, r["code"], np.array(logged)))

HELD = {}
print(f"held-out instances used: {N_HELDOUT} of 32 (n = 200)")
print(f"{'program':8s} {'label':6s} {'replayed gap %':>14s} {'logged, same inst.':>18s} {'max |diff|':>10s} "
      f"{'logged, all 32':>14s} {'wall s':>7s}")
for name, code, logged in held_progs:
    t1 = time.time()
    res = evaluate_set(code, "heldout")
    g = gaps_from_lengths(res["lengths"], refs["heldout"][:N_HELDOUT])
    HELD[name] = {"gaps": g, "mean": float(g.mean()), "logged_same": float(logged[:N_HELDOUT].mean()),
                  "logged_all": float(logged.mean())}
    print(f"{name:8s} {res['label']:6s} {g.mean():14.2f} {logged[:N_HELDOUT].mean():18.2f} "
          f"{np.max(np.abs(g - logged[:N_HELDOUT])):10.1e} {logged.mean():14.2f} {time.time() - t1:7.1f}")
nn_sub = HELD["NN"]["mean"]
for name, h in HELD.items():
    if name != "NN":
        print(f"{name}: closes {100 * (1 - h['mean'] / nn_sub):.0f}% of NN's gap on these {N_HELDOUT} instance(s) "
              f"({100 * (1 - h['logged_all'] / G200_NN):.0f}% on all 32 in the logged run; V1-T needs >= 20%)")

## 9. Results
The table collects, for each arm, the replay (validity and clusters among the matched calls) next to the full-run
numbers: clusters per 1,000 calls, validity, best F, and the held-out gap of the best program.

The figure has four panels:
- **(a)** entropy at the forced position for DTB vs RPB over the whole run;
- **(b)** the behaviour space of the replayed programs (classical MDS of their decision vectors);
- **(c)** full-run discovery curves (distinct behaviour clusters vs LLM calls);
- **(d)** held-out gaps of the arm-seed best programs vs NN and the V1-T bar.

In [ ]:
COLORS = {"T1.0": "tab:blue", "T1.6": "tab:orange", "DTB": "tab:red", "RPB": "tab:purple", "IDEA": "tab:green",
          "NOSEL": "tab:gray", "NN": "black"}
NOSEL = CTX["full_run"]["NOSEL|0"]

# ---------------- summary table
print(f"{'':5s} | {'replay: matched calls, seed 0':^31s} | {'full run: 1,056 calls per arm-seed (seed 0 / seed 1)':^62s}")
print(f"{'arm':5s} | {'calls':>5s} {'valid':>5s} {'p-valid':>7s} {'clust/1k':>9s} | {'clusters/1k':>15s} "
      f"{'validity':>11s} {'best F':>13s} {'best G200 %':>15s}")
print("-" * 104)
for arm in ARMS:
    a = agg.get(("block", arm), Counter())
    f0, f1 = FULL[f"{arm}|0"], FULL[f"{arm}|1"]
    rate = 1000 * demo_c[(arm, 0)] / max(1, demo_N[(arm, 0)])
    print(f"{arm:5s} | {a['calls']:5d} {a['valid']:5d} {a['probe_valid']:7d} {rate:9.1f} | "
          f"{f0['rate_per_1000_calls']:7.1f}/{f1['rate_per_1000_calls']:<7.1f} {f0['validity']:5.2f}/{f1['validity']:<5.2f} "
          f"{f0['best_F']:6.3f}/{f1['best_F']:<6.3f} {f0['G200_best']:7.2f}/{f1['G200_best']:<7.2f}")
print("-" * 104)
print(f"NN (seed program): F = 1, G200 = {G200_NN:.2f}%;  NOSEL best of 252 calls: F = {NOSEL['F']:.3f}, "
      f"G200 = {NOSEL['G200_best']:.2f}%")
print(f"V2-T {v2['verdict']}: DTB/T1.0 = {v2['R']['T1.0']:.3f}, DTB/T1.6 = {v2['R']['T1.6']:.3f}, "
      f"DTB/RPB = {v2['R']['RPB']:.3f}  |  V1-T {v1['verdict']}: all core bests <= {v1['bar_0.8xNN']:.2f}%")
print(f"replay fidelity: labels {tot['label']}/{tot['calls']}, F {tot['F']}/{tot['calls']}, behaviour vectors "
      f"{tot['dec']}/{tot['dec_logged']}, forcings {n_match}/{len(forcing_recs)}, prompts {n_hash}/{n_with}, "
      f"islands {n_ok}/{n_all} identical to the log")

# ---------------- figures
fig, ax = plt.subplots(2, 2, figsize=(16, 12))

# (a) entropy at the forced position, full run
diag = CTX["full_run"]["dtb_rpb_diagnostics"]
bins = np.linspace(0, 3.0, 31)
for arm in ("DTB", "RPB"):
    H = [h for s in SEEDS for h in diag[f"{arm}|{s}"]["H_j_star_values"] if h is not None]
    ax[0, 0].hist(H, bins=bins, alpha=0.55, color=COLORS[arm],
                  label=f"{arm}: {len(H)} branched calls, median H = {np.median(H):.2f} nats")
ax[0, 0].set_yscale("log")
ax[0, 0].set_xlabel("entropy of the top-20 at the forced position j* (nats)")
ax[0, 0].set_ylabel("calls (log scale)")
ax[0, 0].set_title("(a) DTB forces at uncertain tokens, RPB at near-certain ones (full run)")
ax[0, 0].legend()

# (b) behaviour space of the replayed programs
for arm in ARMS + ["NOSEL"]:
    for grp, marker, size in (("block", "o", 45), ("best", "D", 110)):
        idx = [i for i, m in enumerate(pool_meta) if m[1] == arm and m[0] == grp]
        if idx:
            ax[0, 1].scatter(MDS_XY[idx, 0], MDS_XY[idx, 1], c=COLORS[arm], marker=marker, s=size, alpha=0.8,
                             edgecolors="black" if grp == "best" else "none",
                             label=f"{arm} {'matched call' if grp == 'block' else 'arm-seed best'}")
ax[0, 1].scatter(MDS_XY[-1, 0], MDS_XY[-1, 1], c="black", marker="*", s=350, label="NN (seed program)")
for i, m in enumerate(pool_meta):
    if m[0] == "best":
        ax[0, 1].annotate(f"{m[1]}|{m[2]} F={m[4]:.2f}", MDS_XY[i], fontsize=7, xytext=(4, 4), textcoords="offset points")
ax[0, 1].set_xlabel("MDS 1")
ax[0, 1].set_ylabel("MDS 2")
ax[0, 1].set_title(f"(b) behaviour space: {len(X_pool)} programs, {res_pool['n_clusters']} clusters at agreement >= 0.95")
ax[0, 1].legend(fontsize=7, ncol=2, loc="best", framealpha=0.85)

# (c) discovery curves of the full run
for key, v in FULL.items():
    arm, s = key.split("|")
    ax[1, 0].plot(v["discovery_calls"], v["discovery_primary"], color=COLORS[arm], ls="-" if s == "0" else "--",
                  label=f"{arm} seed {s}")
ax[1, 0].set_xlabel("LLM calls")
ax[1, 0].set_ylabel("distinct behaviour clusters (complete linkage, agreement >= 0.95)")
ax[1, 0].set_title("(c) discovery curves (full run): DTB tracks T1.0 and RPB; IDEA leads; T1.6 lags")
ax[1, 0].legend(fontsize=8, ncol=2)

# (d) held-out gaps of the arm-seed bests
keys = [f"{a}|{s}" for s in SEEDS for a in ARMS] + ["NOSEL|0"]
vals = [FULL[k]["G200_best"] if k in FULL else NOSEL["G200_best"] for k in keys]
xs = np.arange(len(keys))
ax[1, 1].bar(xs, vals, color=[COLORS[k.split("|")[0]] for k in keys], alpha=0.8, label="logged, 32 instances")
ax[1, 1].axhline(G200_NN, color="black", lw=1.5, label=f"NN {G200_NN:.2f}%")
ax[1, 1].axhline(0.8 * G200_NN, color="black", ls="--", lw=1.2, label=f"V1-T bar 0.8 x NN = {0.8 * G200_NN:.2f}%")
first = True
for i, k in enumerate(keys):
    if k in HELD:
        ax[1, 1].plot(i, HELD[k]["mean"], "D", color="black", ms=10,
                      label=f"replayed mean, first {N_HELDOUT} instance(s)" if first else None)
        ax[1, 1].plot(i, HELD[k]["logged_same"], "o", color="gold", mec="black", ms=5,
                      label=f"logged mean, same {N_HELDOUT} instance(s)" if first else None)
        first = False
ax[1, 1].set_xticks(xs)
ax[1, 1].set_xticklabels(keys, rotation=45, ha="right")
ax[1, 1].set_ylabel("held-out gap to LKH-3 at n = 200 (%)")
ax[1, 1].set_title("(d) every arm's best beats NN on unseen n = 200 instances (V1-T)")
ax[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## Notes: what is and is not replayed, and how to scale up
- **Replayed exactly:** instance sets, probe states, NN baseline, prompt rendering, DTB/RPB position and token choice,
  island draws, program extraction, sandbox execution (fitness and the 2,000-decision behaviour vectors), the cluster
  rule, the verdict functions, and the held-out gaps.
- **Not replayed:**
  - LLM generation. It needs Qwen3-8B-FP8 on a GPU with vLLM 0.11 (`bash start_vllm.sh compile`, then
    `python method.py search` in the experiment folder), and vLLM sampling is not bit-reproducible across batch
    compositions, so the run's statistics reproduce from its logs rather than from fresh generations.
  - The corpus and judge stages (V3-T).
  - The round-block bootstraps over all 33 rounds.
  - The extended (96) and TSPLIB (13) held-out sets.
- **To scale up:** raise `N_CALLS_PER_ARM` (max 16 in the demo data), `N_BEST_PROGRAMS` (max 11), `N_HELDOUT` (max 32,
  the original) and `N_HELDOUT_PROGRAMS` in the configuration cell. The complete analysis needs all 10,812 logged rows
  and their decision vectors (`data/calls.jsonl`, `data/probe_decisions.bin`) in the experiment folder. There,
  `python method.py analyze` re-derives every number above, and `python method.py rederive` checks it on an
  independent, standard-library code path.